# Reproducing de Paula et al. (2026) from DANDI:000713

Paper. E. V. de Paula, R. M. Jungmann, A. J. Fontenele, L. A. A. Aguiar, P. V. Carelli, F. S. Matias, M. Copelli and N. A. P. de Vasconcelos, "Revealing Stimulus-Dependent Dynamics Through Statistical Complexity", PRX Life 4, 023018 (2026), doi:10.1103/wndc-k654.

Dataset. DANDI:000713, "Allen Institute - Visual Behavior - Neuropixels", version 0.240702.1725 (doi:10.48324/dandi.000713/0.240702.1725). This is the version the paper cites (its Ref. 22, as the claims file gives it), and it is the only published version of the dandiset; the draft has the same 4293 assets and was last modified on 2024-07-02, when this version was published. The 153 ecephys session files are streamed from the archive; nothing is written to disk.

Text. The claims file `claims/paula-2026.yaml` was written from the version of record (corrected 22 July 2026) and its Supplemental Material. The paper links no analysis code.

Claims. All 38 claims of the claims file are tested here; none is blocked. Claim `fig3a-reference-systems` uses synthetic series only (`data: elsewhere`, no recordings). The outcomes are in the attempt record and the summary at the end.

| Claim | Tested here |
|---|---|
| table1-sample | tested |
| fig1c-cv-region-medians | tested |
| fig1c-cv-above-surrogate | tested |
| fig1d-surrogate-abolishes-cv-correlations | tested |
| fig1e-positive-cv-modules | tested |
| fig1e-dg-ca3-anticorrelation-hubs | tested |
| fig1e-midbrain-weakest-cv-correlations | tested |
| fig2a-vis-cv-by-condition | tested |
| fig2a-thalamus-cv-by-condition | tested |
| fig2a-hippocampus-cv-stable | tested |
| fig2a-midbrain-cv-invariant | tested |
| fig2b-vis-summed-js-cv | tested |
| fig2b-hp-mb-summed-js-cv | tested |
| fig2b-visually-driven-js-cv-low | tested |
| fig2b-dg-ca3-high-js-cv | tested |
| fig2b-legend-js-cv | tested |
| fig3a-reference-systems | tested (synthetic series) |
| fig3b-complexity-levels-by-region | tested |
| fig3c-vis-ch-plane | tested |
| fig3d-th-ch-plane | tested |
| fig3e-hp-ch-plane | tested |
| fig3f-mb-ch-plane | tested |
| fig3-complexity-range-fractions | tested |
| fig4a-vis-complexity-by-condition | tested |
| fig4b-thalamus-lgd-lp-modulated | tested |
| fig4b-other-thalamus-not-modulated | tested |
| fig4c-dg-ca3-complexity | tested |
| fig4d-midbrain-complexity-invariant | tested |
| fig4e-js-complexity-natural-images | tested |
| fig4fg-js-complexity-active-blanks | tested |
| fig4h-js-complexity-spontaneous-converges | tested |
| supp-robustness-parameters | tested |
| fig5-vis-complexity-cv-coupling | tested |
| fig5-th-weaker-coupling | tested |
| fig5-hp-spontaneous-elevated | tested |
| fig5-mb-flat | tested |
| supp-fig6-mean-correlation | tested |
| complexity-separates-images-from-spontaneous | tested |

Method. The helper module `paula.py` reads each session and computes, for every local population (the units of one subregion in one session with at least 30 units) and every 100-s window of the four conditions, the CV, the permutation entropy H and the statistical complexity C of the population rate, the same for an ISI-shuffled surrogate, the mean pairwise correlation (visual cortex and midbrain), and C and H for the parameter combinations of the supplement. Its docstring says where each parameter comes from. Every claim's section below starts from that table.

In [ ]:
import os
for v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"  # set before numpy loads: the four worker processes each run many small fits
import time, zlib, warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import paula

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
print("dandiset", paula.DANDISET, "version", paula.VERSION)

## Loading

One pass over all ecephys sessions, four sessions at a time in separate processes. Each session's spike times are read up to the end of its spontaneous epoch. This takes about 35 minutes.

In [ ]:
t0 = time.time()
sessions, n_assets = paula.list_sessions()
metas, windows, cv10_list, failed = paula.run_all(sessions, workers=4)
t_extract = time.time() - t0
print(f"{len(sessions)} session files of {n_assets} assets; extraction took {t_extract / 60:.1f} min")

In [ ]:
import resource
print("sessions read:", len(metas), "failed:", len(failed), failed)
print("subjects:", metas.subject.nunique(), "| session types:", metas.session_description.value_counts().to_dict())
print("genotypes:", metas.genotype.value_counts().to_dict())
print("units per session: median", metas.n_units.median(), "range", metas.n_units.min(), "to", metas.n_units.max())
print("active block length (s):", (metas.active_end - metas.active_start).describe().round(1).to_dict())
print("active windows per session:", metas.n_active_windows.value_counts().to_dict())
print("spontaneous epochs after the task (s):", pd.Series([e[-1][1] - e[-1][0] for e in metas.spont_epochs]).describe().round(1).to_dict())
print("images per session:", metas.n_images.value_counts().to_dict(), "| flashes shown per session: median", metas.n_flashes.median(),
      "| omitted: median", metas.n_omitted.median(), "| change flashes: median", metas.n_changes.median())
sl = windows[windows.units == "all"].groupby(["condition", "reading"]).series_length.describe()[["min", "50%", "max"]].round(1)
print("length of the series each window is cut from (s):\n", sl)
print("window rows by unit reading and condition:\n", windows.groupby(["units", "condition", "reading"]).size())
for col, unit in [("cv", ""), ("H", ""), ("C", ""), ("rate", "spikes/s per unit"), ("corr50", ""), ("corr20", "")]:
    x = windows[col]
    print(f"{col}: {np.nanmin(x):.3f} to {np.nanmax(x):.3f} {unit}, NaN {int(x.isna().sum())}")
# recordings deposited more than once: identical spike-time fingerprints or spike counts
fp = metas.fingerprint.apply(lambda v: tuple(np.round(v, 6)))
print("sessions sharing a spike-time fingerprint:", int(fp.duplicated(keep=False).sum()),
      "| sharing a total spike count:", int(metas.n_spikes.duplicated(keep=False).sum()),
      "| sharing subject and start time:", int(metas.duplicated(["subject", "session_start_time"], keep=False).sum()))
print("peak memory of this process (GB):", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6)

### Clock check

The stimulus table and the spike times must share a clock. For every session the notebook counts the visual cortex spikes 30 to 130 ms after each flash onset minus those in the 100 ms before, for the true onsets and for onsets shifted by 375 ms into the gray screen, and for the omitted flashes (no image appears, so no response is expected).

In [ ]:
cl = metas[["clock_true", "clock_shift", "clock_omitted", "n_vis_units_clock"]]
print(cl.describe().round(2))
print("sessions where the response to true onsets exceeds that to shifted onsets:", int((cl.clock_true > cl.clock_shift).sum()), "of", len(cl))
print("sessions where the true-onset response exceeds the omitted-flash response:", int((cl.clock_true > cl.clock_omitted).sum()), "of", len(cl))
print("smallest true-onset contrast:", round(cl.clock_true.min(), 2), "| largest shifted contrast:", round(cl.clock_shift.max(), 2))

## table1-sample

> The analysis rests on 102 sessions and 1437 local populations (VIS 586, TH 313, HP 363, MB 175), with 53 985 visual cortex, 20 541 thalamic, 42 396 hippocampal and 17 163 midbrain neurons, and 77 216 analysis windows (11 321 natural images, 51 704 active behavior, 11 321 blanks, 2870 spontaneous).

Counts under each unit reading ('all' units; 'qc', ISI violations < 0.5, amplitude cutoff < 0.1, presence ratio > 0.9) and each session reading ((a) sessions with at least one kept population; (b) those that also have the spontaneous gray-screen epoch). The selection rule (at least 30 units, in at least eight sessions) is also applied to every location label of the files, to see which labels it keeps beyond Table I.

In [ ]:
TABLE1 = {"VISal": 8572, "VISam": 8566, "VISl": 9255, "VISp": 10010, "VISpm": 10250, "VISrl": 7332,
          "LGd": 2169, "LP": 2795, "MGd": 2497, "MGm": 1928, "MGv": 4492, "PIL": 868, "POL": 1298, "PoT": 1461, "SGN": 3033,
          "CA1": 22399, "CA3": 3729, "DG": 7486, "POST": 1367, "ProS": 3067, "SUB": 4348,
          "APN": 10716, "NB": 793, "NOT": 309, "MRN": 3661, "SCig": 1684}
EXP1 = {"sessions": 102, "VIS populations": 586, "TH populations": 313, "HP populations": 363, "MB populations": 175,
        "VIS neurons": 53985, "TH neurons": 20541, "HP neurons": 42396, "MB neurons": 17163,
        "natural image windows": 11321, "active behavior windows": 51704, "blank windows": 11321, "spontaneous windows": 2870}
has_spont = set(metas.session_index[metas.n_spont_windows >= 2])
# labels kept by the rule over all location labels, from the unit counts of each session
for ur in ["all", "qc"]:
    rows = [(m.session_index, r, c) for m in metas.itertuples() for r, c in getattr(m, f"region_counts_{ur}").items()]
    rc = pd.DataFrame(rows, columns=["session", "region", "n"])
    rc = rc[rc.n >= paula.MIN_UNITS]
    ns = rc.groupby("region").session.nunique()
    lab = sorted(ns[ns >= paula.MIN_SESSIONS].index)
    print(f"units={ur}: labels kept by the rule: {len(lab)}; not in Table I: {[l for l in lab if l not in paula.REGION_AREA]};",
          f"Table I subregions not kept: {[r for r in paula.REGION_ORDER if r not in lab]}")
t1 = {}
for ur in ["all", "qc"]:
    regs = paula.kept_regions(windows, ur)
    for sr in ["a", "b"]:
        w = paula.conditions(windows, ur, "a", "a")
        w = w[w.region.isin(regs)]
        if sr == "b":
            w = w[w.session.isin(has_spont)]
        pops = w[w.condition == "active"].drop_duplicates(["session", "region"])
        res = {"sessions": pops.session.nunique()}
        for a in paula.AREAS:
            res[f"{a} populations"] = int((pops.area == a).sum())
        for a in paula.AREAS:
            res[f"{a} neurons"] = int(pops.loc[pops.area == a, "n"].sum())
        res["natural image windows"] = int((w.condition == "images").sum())
        res["active behavior windows"] = int((w.condition == "active").sum())
        res["blank windows"] = int((w.condition == "blanks").sum())
        res["spontaneous windows"] = int((w.condition == "spont").sum())
        t1[f"{ur}/{sr}"] = res
        if sr == "a":
            per = pops.groupby("region").agg(populations=("n", "size"), neurons=("n", "sum")).reindex(paula.REGION_ORDER)
            per["Table I neurons"] = pd.Series(TABLE1)
            per["ratio"] = (per.neurons / per["Table I neurons"]).round(2)
            print(f"\nunits={ur}: per subregion (kept regions: {len(regs)})\n", per.T.to_string())
tab = pd.DataFrame(t1)
tab["expected"] = pd.Series(EXP1)
tab["tolerance"] = tab["expected"] * 0.1
for c in t1:
    tab["within " + c] = (tab[c] - tab["expected"]).abs() <= tab["tolerance"]
print(tab.to_string())
for c in t1:
    print(c, "all within tolerance:", bool(tab["within " + c].all()))
# windows per population and session reading do not depend on the blank or spontaneous reading
print("blank windows under reading b:", int((paula.conditions(windows, "all", "b", "b").condition == "blanks").sum()))

**Result: not reproduced.** Under neither unit reading nor either session reading do the counts match: every one of the 153 ecephys sessions of the dandiset has kept populations and a spontaneous epoch, so both session readings give 153 sessions, not 102, and the population and window counts are about 1.5 times the paper's under the 'all' reading. The 'all' reading keeps exactly the 26 subregions of Table I among the labels of Table I (the rule also keeps six labels Table I does not list), while 'qc' drops at least one of them. Neuron counts per population are larger than Table I's under 'all' and smaller under 'qc' (ratios printed above). The 'all' reading is adopted by the other claims as an assumption (it keeps Table I's subregion list and is the closer of the two), with 'qc' shown as a variant where it bears on a claim.

### Readings used below

The unit reading is 'all' (assumption, above). The four condition readings combine the blank reading (a: after every shown flash; b: only after flashes followed by another flash) with the spontaneous reading (a: the first two 100-s windows of the gray screen; b: two windows centred in it). They are written `blanks/spont`, for example `a/b`.

In [ ]:
REG = paula.kept_regions(windows, "all")
CR = [("a", "a"), ("a", "b"), ("b", "a"), ("b", "b")]
REGQ = paula.kept_regions(windows, "qc")
W = {}
WQ = {}
for b, s in CR:
    x = paula.conditions(windows, "all", b, s)
    W[f"{b}/{s}"] = x[x.region.isin(REG)]
    x = paula.conditions(windows, "qc", b, s)
    WQ[f"{b}/{s}"] = x[x.region.isin(REGQ)]
ACT = W["a/a"][W["a/a"].condition == "active"]
ACTQ = WQ["a/a"][WQ["a/a"].condition == "active"]
print(len(REG), "subregions;", ACT.drop_duplicates(["session", "region"]).shape[0], "populations; active windows", len(ACT))

def within(x, lo, hi):
    return (x >= lo - 1e-12) & (x <= hi + 1e-12)

def widen(lo, hi):
    return lo - 0.1 * abs(lo), hi + 0.1 * abs(hi)

def tol(x, v, t):
    return abs(x - v) <= t + 1e-12

## fig1c-cv-region-medians

> During active behavior visual cortex subregions have the highest population CV (medians 0.40 to 0.59), thalamic subregions moderate (0.27 to 0.37), hippocampal subregions high (0.25 to 0.49, dentate gyrus highest at 0.49) and midbrain the lowest (0.16 to 0.45).

In [ ]:
RANGES_CV = {"VIS": (0.40, 0.59), "TH": (0.27, 0.37), "HP": (0.25, 0.49), "MB": (0.16, 0.45)}
IQR_CV = {"VIS": (0.12, 0.18), "TH": (0.10, 0.15), "HP": (0.08, 0.13), "MB": (0.05, 0.23)}

def cv_medians(act):
    g = act.groupby("region").cv
    t = pd.DataFrame({"median": g.median(), "iqr": g.quantile(0.75) - g.quantile(0.25), "windows": g.size()}).reindex(REG_OF(act))
    t["area"] = [paula.REGION_AREA[r] for r in t.index]
    return t

def REG_OF(d):
    return [r for r in paula.REGION_ORDER if r in set(d.region)]

def judge_fig1c(t, label):
    ok_all, ok_iqr = True, True
    for a, (lo, hi) in RANGES_CV.items():
        x = t[t.area == a]
        wl, wh = widen(lo, hi)
        inside = within(x["median"], wl, wh)
        ext = tol(x["median"].min(), lo, 0.1 * lo) and tol(x["median"].max(), hi, 0.1 * hi)
        il, ih = widen(*IQR_CV[a])
        iqr_in = within(x["iqr"], il, ih)
        print(f"{label} {a}: medians {x['median'].min():.3f} to {x['median'].max():.3f} (expected {lo} to {hi}); inside widened range {int(inside.sum())}/{len(x)}; "
              f"extremes within tolerance {ext}; IQR {x['iqr'].min():.3f} to {x['iqr'].max():.3f}, inside {int(iqr_in.sum())}/{len(x)}")
        ok_all &= bool(inside.all()) and ext
        ok_iqr &= bool(iqr_in.all())
    hp = t[t.area == "HP"]["median"]
    print(f"{label}: highest hippocampal median is {hp.idxmax()} ({hp.max():.3f}); medians and extremes all match: {ok_all}; IQRs all match: {ok_iqr}")
    return ok_all and hp.idxmax() == "DG", ok_iqr

t = cv_medians(ACT)
print(t.round(3).T.to_string())
res_main = judge_fig1c(t, "units=all")
tq = cv_medians(ACTQ)
print(tq.round(3).T.to_string())
res_q = judge_fig1c(tq, "units=qc (variant)")
# by session group: the paper's sessions are not identified, so the result is also given for familiar and novel image sets
fam = set(metas.session_index[metas.session_description.str.contains("_G_")])
for name, ss in [("image set G", fam), ("image set H", set(metas.session_index) - fam)]:
    x = ACT[ACT.session.isin(ss)].groupby("area").cv.median()
    print(name, "area medians:", x.round(3).to_dict())

**Result: not reproduced.** Visual cortex matches (medians 0.426 to 0.644, extremes within tolerance) and DG has the highest hippocampal median, but the lowest thalamic median (PIL 0.306) is above 0.27 +/- 0.027, the highest hippocampal (DG 0.564) above 0.49 +/- 0.049 and the highest midbrain (NB 0.564) above 0.45 +/- 0.045; the IQRs fall inside their ranges in 16 of 26 subregions. The 'qc' variant misses too (highest thalamic median LP 0.414). The outcome rests on the unit-selection assumption.

## fig1c-cv-above-surrogate

> Empirical CV exceeds that of ISI-shuffled surrogates in every brain region (VIS 2 to 3 times, surrogate medians 0.21 to 0.24; TH 1.3 to 1.6; HP 1.5 to 2.2; MB 1.3 to 1.9), with Mann-Whitney p < 0.001 for all regions.

One surrogate per active window (Methods II.E). The surrogate is drawn under the seed of this claim, `zlib.crc32(b"paula-2026/fig1c-cv-above-surrogate")`, and under that seed plus 1000, 2000, 3000 and 4000; the same surrogates serve the claims of Figs. 1d and 3 (see the deviations).

In [ ]:
RATIO = {"VIS": (2, 3), "TH": (1.3, 1.6), "HP": (1.5, 2.2), "MB": (1.3, 1.9)}
SFX = ["", "_s1000", "_s2000", "_s3000", "_s4000"]

def surrogate_check(act, sfx, verbose):
    g = act.groupby("region")
    t = pd.DataFrame({"emp": g.cv.median(), "surr": g["s_cv" + sfx].median()}).reindex(REG_OF(act))
    t["ratio"] = t.emp / t.surr
    t["area"] = [paula.REGION_AREA[r] for r in t.index]
    p_sub = {r: stats.mannwhitneyu(x.cv.dropna(), x["s_cv" + sfx].dropna()).pvalue for r, x in g}
    t["p"] = pd.Series(p_sub)
    ok_ratio = True
    for a, (lo, hi) in RATIO.items():
        wl, wh = widen(lo, hi)
        x = t[t.area == a]
        ok_ratio &= bool(within(x.ratio, wl, wh).all())
        if verbose:
            print(f"{a}: ratios {x.ratio.min():.2f} to {x.ratio.max():.2f} (expected {lo} to {hi}, widened {wl:.2f} to {wh:.2f}); inside {int(within(x.ratio, wl, wh).sum())}/{len(x)}")
    vs = t[t.area == "VIS"].surr
    ok_vs = bool(within(vs, 0.189, 0.264).all())
    pa = {}
    for a, x in act.groupby("area"):
        pa[a] = (stats.mannwhitneyu(x.cv.dropna(), x["s_cv" + sfx].dropna()).pvalue, x.cv.median() > x["s_cv" + sfx].median())
    ok_sub = bool((t.p < 0.001).all() and (t.emp > t.surr).all())
    ok_area = all(p < 0.001 and hi for p, hi in pa.values())
    if verbose:
        print(t.round(4).T.to_string())
        print(f"VIS surrogate medians {vs.min():.3f} to {vs.max():.3f} (0.189 to 0.264): {ok_vs}")
        print("per area Mann-Whitney:", {a: (f"{p:.1e}", bool(h)) for a, (p, h) in pa.items()})
        print(f"tests hold per subregion: {ok_sub}; per area: {ok_area}; all ratios inside: {ok_ratio}; largest subregion p {t.p.max():.1e}")
    return t, ok_ratio, ok_vs, ok_sub, ok_area

base = surrogate_check(ACT, "", True)
print("\nfive seeds (claim seed, +1000, +2000, +3000, +4000):")
for s in SFX:
    t_, r_, v_, ps_, pa_ = surrogate_check(ACT, s, False)
    print(f"seed{s or ' (claim)'}: ratio range {t_.ratio.min():.3f} to {t_.ratio.max():.3f}; VIS surrogate medians {t_[t_.area=='VIS'].surr.min():.4f} to {t_[t_.area=='VIS'].surr.max():.4f}; "
          f"ratios inside {r_}; VIS surrogates inside {v_}; tests per subregion {ps_}, per area {pa_}; largest p {t_.p.max():.1e}")
print("\nvariant units=qc:")
_ = surrogate_check(ACTQ, "", True)

**Result: partially reproduced.** Empirical CV is above the surrogate CV in every subregion and every area, Mann-Whitney p below 1e-72 in all 26 subregions and p = 0 at printed precision for the four areas, and the visual surrogate medians (0.226 to 0.243) lie in 0.189 to 0.264. The ratios lie in their area's widened range in 24 of 26 subregions: DG (3.01) and POST (1.32) fall outside the hippocampal 1.35 to 2.42. All five seeds give the same result (ratios 1.314 to 3.009). The outcome rests on the unit-selection assumption.

## fig1d-surrogate-abolishes-cv-correlations

> Inter-regional correlations of CV time series during active behavior are present in the data and abolished in the ISI-shuffled surrogates.

Per pair of subregions recorded in the same session, the Pearson correlation of their 10-s CV series over the active windows (360 values per session at most). Reading (a) concatenates the series of all shared sessions; reading (b) averages the per-session correlations.

In [ ]:
cvs = pd.DataFrame(cv10_list)
cvs = cvs[(cvs.units == "all") & cvs.region.isin(REG)].sort_values(["session", "region", "k"])
SCOLS = ["s_cv10" + x for x in SFX]
series = {}
for (s, r), g in cvs.groupby(["session", "region"]):
    series[(s, r)] = [np.concatenate(list(g.cv10))] + [np.concatenate(list(g[c])) for c in SCOLS]
by_sess = {}
for (s, r) in series:
    by_sess.setdefault(s, []).append(r)

def corr_mats(which):
    n = len(REG)
    ca, cb, ns = (np.full((n, n), np.nan) for _ in range(3))
    for i in range(n):
        for j in range(i + 1, n):
            xs, ys, rs = [], [], []
            for s, rr in by_sess.items():
                if REG[i] in rr and REG[j] in rr:
                    x, y = series[(s, REG[i])][which], series[(s, REG[j])][which]
                    ok = np.isfinite(x) & np.isfinite(y)
                    if ok.sum() > 2 and x[ok].std() > 0 and y[ok].std() > 0:
                        xs.append(x[ok]); ys.append(y[ok]); rs.append(np.corrcoef(x[ok], y[ok])[0, 1])
            if xs:
                ca[i, j] = ca[j, i] = np.corrcoef(np.concatenate(xs), np.concatenate(ys))[0, 1]
                cb[i, j] = cb[j, i] = np.mean(rs)
                ns[i, j] = ns[j, i] = len(rs)
    f = lambda m: pd.DataFrame(m, index=REG, columns=REG)
    return f(ca), f(cb), f(ns)

EMP_A, EMP_B, NSH = corr_mats(0)
SUR_A, SUR_B, _ = corr_mats(1)
iu = np.triu_indices(len(REG), 1)
print("pairs with at least one shared session:", int(np.isfinite(EMP_A.values[iu]).sum()), "of", len(iu[0]),
      "| shared sessions per pair: median", np.nanmedian(NSH.values[iu]))
ok1d = {}
for lab, e, s in [("a", EMP_A, SUR_A), ("b", EMP_B, SUR_B)]:
    ev, sv = e.values[iu], s.values[iu]
    ev, sv = ev[np.isfinite(ev)], sv[np.isfinite(sv)]
    q = np.percentile(np.abs(sv), [50, 90, 99, 100])
    ok1d[lab] = (ev.max() >= 0.45) and (np.abs(sv).max() < 0.2)
    print(f"reading {lab}: empirical r {ev.min():.3f} to {ev.max():.3f}, pairs >= 0.45: {int((ev >= 0.45).sum())}; "
          f"surrogate |r| median {q[0]:.3f}, 90th {q[1]:.3f}, 99th {q[2]:.3f}, max {q[3]:.3f}; pairs with |r| > 0.2: {int((np.abs(sv) > 0.2).sum())}; passes {ok1d[lab]}")
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
for a_, (lab, e, s) in zip(ax, [("a", EMP_A, SUR_A), ("b", EMP_B, SUR_B)]):
    m = np.tril(e.values, -1) + np.triu(s.values, 1)
    im = a_.imshow(m, vmin=-0.7, vmax=0.7, cmap="RdBu_r")
    a_.set_xticks(range(len(REG)), REG, rotation=90, fontsize=7); a_.set_yticks(range(len(REG)), REG, fontsize=7)
    a_.set_title(f"reading {lab}: lower empirical, upper surrogate")
plt.colorbar(im, ax=ax); plt.show()
for k, sfx in enumerate(SFX[1:], start=2):
    sa, sb, _ = corr_mats(k)
    va, vb = np.abs(sa.values[iu]), np.abs(sb.values[iu])
    print(f"surrogate seed{sfx}: max |r| reading a {np.nanmax(va):.3f} (pairs > 0.2: {int((va > 0.2).sum())}), reading b {np.nanmax(vb):.3f} (pairs > 0.2: {int((vb > 0.2).sum())})")
print("colour scale -0.7 to 0.7; values outside it:", int((np.abs(np.r_[EMP_A.values[iu], EMP_B.values[iu]]) > 0.7).sum()))

**Result: not reproduced.** The empirical correlations are present (largest 0.772 under reading a and 0.669 under b), but the surrogate ones are not abolished: 115 of 325 pairs (reading a) and 213 (reading b) have |r| above 0.2, largest 0.791 and 0.662. The ISI shuffle keeps each window's spike count, so the surrogate CV still follows the rate of each 100-s window, and those rates co-vary across regions. The outcome rests on the unit-selection assumption.

## fig1e-positive-cv-modules

> The strongest 10% of positive CV correlations (0.50 to 0.67) link subregions within functional modules (visual areas together, subicular regions together, thalamic populations together), with connections between distinct systems largely absent.

In [ ]:
def pairs_table(m):
    rows = [(REG[i], REG[j], m.values[i, j]) for i, j in zip(*iu) if np.isfinite(m.values[i, j])]
    p = pd.DataFrame(rows, columns=["r1", "r2", "r"])
    p["same_area"] = [paula.REGION_AREA[a] == paula.REGION_AREA[b] for a, b in zip(p.r1, p.r2)]
    return p

ok1e = {}
for lab, m in [("a", EMP_A), ("b", EMP_B)]:
    p = pairs_table(m)
    top = p.nlargest(int(round(0.1 * len(p))), "r")
    lo_ok, hi_ok = tol(top.r.min(), 0.50, 0.05), tol(top.r.max(), 0.67, 0.067)
    frac = top.same_area.mean()
    ok1e[lab] = lo_ok and hi_ok and frac > 0.5
    print(f"reading {lab}: top {len(top)} of {len(p)} pairs, r {top.r.min():.3f} to {top.r.max():.3f} (expected 0.50 to 0.67; low end within tolerance {lo_ok}, high end {hi_ok}); "
          f"same major area {int(top.same_area.sum())}/{len(top)} ({frac:.2f}); passes {ok1e[lab]}")
    print("   ", "; ".join(f"{a}-{b} {r:.2f}" for a, b, r in top[["r1", "r2", "r"]].values))

**Result: partially reproduced.** The strongest 10% of pairs join subregions of the same major area in most cases (21 of 32 under reading a, 26 of 32 under b), mostly visual areas with each other and thalamic or subicular pairs. The range misses at one end under each reading: 0.459 to 0.772 under a (high end above 0.737), 0.450 to 0.669 under b (low end at the edge of 0.45, below it before rounding). The outcome rests on the unit-selection assumption.

## fig1e-dg-ca3-anticorrelation-hubs

> Among the 10% most anticorrelated region pairs (r from -0.27 to -0.002), DG and CA3 are the hubs of negative CV relationships with other regions.

In [ ]:
ok1h = {}
for lab, m in [("a", EMP_A), ("b", EMP_B)]:
    p = pairs_table(m)
    neg = p.nsmallest(int(round(0.1 * len(p))), "r")
    cnt = pd.Series(list(neg.r1) + list(neg.r2)).value_counts()
    top2 = set(cnt.index[:2])
    tie = cnt.iloc[1] == cnt.iloc[2] if len(cnt) > 2 else False
    ok1h[lab] = top2 == {"DG", "CA3"} and tol(neg.r.min(), -0.27, 0.027)
    print(f"reading {lab}: {len(neg)} pairs, r {neg.r.min():.3f} to {neg.r.max():.3f} (most negative expected -0.27 +/- 0.027); "
          f"appearances {cnt.head(6).to_dict()}; tie for second place {tie}; passes {ok1h[lab]}")

**Result: partially reproduced.** Under reading b, CA3 (17) and DG (14) are the subregions in most of the 32 most negative pairs, but the most negative value is -0.178, not -0.27 +/- 0.027. Under reading a, NB leads (15) and the most negative value is -0.496. The outcome rests on the unit-selection assumption.

## fig1e-midbrain-weakest-cv-correlations

> Midbrain areas have the weakest CV correlations with all other regions, near zero (-0.04 to +0.03).

In [ ]:
ok1m = {}
lo_n, hi_n = widen(-0.04, 0.03)
for lab, m in [("a", EMP_A), ("b", EMP_B)]:
    mm = m.abs().mean(axis=1)
    area_mean = mm.groupby([paula.REGION_AREA[r] for r in mm.index]).mean()
    p = pairs_table(m)
    mbp = p[[paula.REGION_AREA[a] == "MB" or paula.REGION_AREA[b] == "MB" for a, b in zip(p.r1, p.r2)]]
    near = within(mbp.r, lo_n, hi_n).mean()
    ok1m[lab] = area_mean.idxmin() == "MB" and near > 0.5
    print(f"reading {lab}: mean |r| by area {area_mean.round(3).to_dict()}; per MB subregion {mm[[r for r in REG if paula.REGION_AREA[r]=='MB']].round(3).to_dict()}; "
          f"MB pairs in {lo_n:.3f} to {hi_n:.3f}: {int(within(mbp.r, lo_n, hi_n).sum())}/{len(mbp)} ({near:.2f}); passes {ok1m[lab]}")

**Result: partially reproduced.** Under reading b the midbrain has the lowest mean |r| of the four areas (0.201 against 0.235 to 0.334), but only 2 of its 115 pairs lie in -0.044 to 0.033. Under reading a, hippocampus is lowest (0.179 against midbrain 0.186) and 16 of 115 midbrain pairs are near zero. The outcome rests on the unit-selection assumption.

## Medians of CV by condition (claims of Fig. 2a)

The median CV of each subregion in each condition, under each condition reading. Rows are subregions; columns condition and reading.

In [ ]:
def med_table(value, Wd=W):
    out = {}
    for k, w in Wd.items():
        out[k] = w.groupby(["region", "condition"])[value].median().unstack()[paula.CONDITIONS]
    return pd.concat(out, axis=1).reindex([r for r in paula.REGION_ORDER if r in set(Wd["a/a"].region)])

MED_CV = med_table("cv")
print(MED_CV.round(3).to_string())
print("windows per condition (reading a/a):", W["a/a"].condition.value_counts().to_dict())

def check_expected(med, exp, readings=None):
    """exp: list of (label, region, condition(s), kind, value, tolerance); returns per reading the passes."""
    res = {}
    for rd in (readings or list(med.columns.levels[0])):
        rows = []
        for lab, reg, conds, kind, v, t in exp:
            vals = med.loc[reg, rd][conds] if isinstance(conds, list) else pd.Series({conds: med.loc[reg, (rd, conds)]})
            x = {"min": vals.min(), "max": vals.max(), "each": None, "value": vals.iloc[0]}[kind] if kind != "each" else None
            if kind == "each":
                ok = bool(all(tol(y, v, t) for y in vals))
                x = f"{vals.min():.3f}-{vals.max():.3f}"
            else:
                ok = tol(x, v, t)
                x = round(float(x), 3)
            rows.append((lab, x, v, ok))
        res[rd] = rows
    return res

def show_expected(res):
    for rd, rows in res.items():
        n = sum(r[3] for r in rows)
        print(f"reading {rd}: {n}/{len(rows)} within tolerance; " + "; ".join(f"{l} {x} ({v})" + ("" if ok else " MISS") for l, x, v, ok in rows))

## fig2a-vis-cv-by-condition

> In visual cortex, median CV is higher in active behavior and blank screens (VISp 0.59 and 0.55, VISl 0.56 and 0.46, VISal 0.55 and 0.46) than in spontaneous activity (0.29, 0.30, 0.30), with natural images intermediate (0.39, 0.46, 0.48).

In [ ]:
E2A = []
for r, vals in {"VISp": (0.59, 0.55, 0.29, 0.39), "VISl": (0.56, 0.46, 0.30, 0.46), "VISal": (0.55, 0.46, 0.30, 0.48)}.items():
    for c, v in zip(["active", "blanks", "spont", "images"], vals):
        E2A.append((f"{r} {c}", r, c, "value", v, 0.1 * v))
r2a = check_expected(MED_CV, E2A)
show_expected(r2a)
for rd in r2a:
    m = MED_CV[rd]
    order = {r: bool(m.loc[r, "active"] > m.loc[r, "images"] > m.loc[r, "spont"] and m.loc[r, "blanks"] > m.loc[r, "spont"]) for r in ["VISp", "VISl", "VISal"]}
    print(f"reading {rd}: order active > images > spont and blanks > spont: {order}")
print("variant units=qc:")
MED_CVQ = med_table("cv", WQ)
show_expected(check_expected(MED_CVQ, E2A, ["a/a"]))

**Result: reproduced.** Under readings a/a and b/a all twelve medians are within tolerance (for example VISp 0.644, 0.602, 0.316 and 0.418 against 0.59, 0.55, 0.29 and 0.39), and the order active > images > spontaneous with blanks > spontaneous holds in the three areas under every reading. Under the 'qc' variant 8 of 12 are within tolerance. The outcome rests on the unit-selection assumption.

## fig2a-thalamus-cv-by-condition

> Thalamic populations show the same pattern with smaller effects, LP median CV 0.37 in active behavior against 0.28 in spontaneous activity, LGd 0.32 against 0.23.

In [ ]:
E2T = [("LP active", "LP", "active", "value", 0.37, 0.037), ("LP spont", "LP", "spont", "value", 0.28, 0.028),
       ("LGd active", "LGd", "active", "value", 0.32, 0.032), ("LGd spont", "LGd", "spont", "value", 0.23, 0.023)]
show_expected(check_expected(MED_CV, E2T))
for rd in MED_CV.columns.levels[0]:
    m = MED_CV[rd]
    d = (m["active"] - m["spont"])
    print(f"reading {rd}: active minus spontaneous LP {d['LP']:.3f}, LGd {d['LGd']:.3f}; VISp {d['VISp']:.3f}, VISl {d['VISl']:.3f}, VISal {d['VISal']:.3f}; "
          f"smaller in both thalamic nuclei than in all three visual areas: {bool(max(d['LP'], d['LGd']) < min(d['VISp'], d['VISl'], d['VISal']))}")

**Result: reproduced.** Under every reading the four medians are within tolerance (LP 0.396 and 0.286, LGd 0.330 and 0.226 against 0.37, 0.28, 0.32 and 0.23), and the active minus spontaneous difference of LP (0.110) and LGd (0.104) is below that of VISp, VISl and VISal (0.271 to 0.327). The outcome rests on the unit-selection assumption.

## fig2a-hippocampus-cv-stable

> DG and CA3 keep nearly constant median CV across the visually driven conditions (DG 0.48 to 0.50, CA3 0.41 to 0.42; CA1 0.25), with modest increases in spontaneous activity (DG 0.51, CA3 0.43).

In [ ]:
VD = ["images", "active", "blanks"]
E2H = [("DG VD low", "DG", VD, "min", 0.48, 0.048), ("DG VD high", "DG", VD, "max", 0.50, 0.05),
       ("CA3 VD low", "CA3", VD, "min", 0.41, 0.041), ("CA3 VD high", "CA3", VD, "max", 0.42, 0.042),
       ("CA1 VD each", "CA1", VD, "each", 0.25, 0.025),
       ("DG spont", "DG", "spont", "value", 0.51, 0.051), ("CA3 spont", "CA3", "spont", "value", 0.43, 0.043)]
show_expected(check_expected(MED_CV, E2H))
for rd in MED_CV.columns.levels[0]:
    m = MED_CV[rd]
    print(f"reading {rd}: spontaneous above every visually driven median: DG {bool(m.loc['DG','spont'] > m.loc['DG', VD].max())}, CA3 {bool(m.loc['CA3','spont'] > m.loc['CA3', VD].max())}")

**Result: partially reproduced.** The pattern holds: DG and CA3 vary little across the visually driven conditions (DG 0.555 to 0.576, CA3 0.449 to 0.458), CA1 is 0.263 to 0.266, and the spontaneous medians of DG and CA3 exceed their visually driven ones under every reading. CA3 and CA1 are within tolerance, and CA3 spontaneous (0.471) under the spontaneous reading b, but DG is higher than the paper's values (0.555 to 0.580 against 0.48 to 0.51). The outcome rests on the unit-selection assumption.

## fig2a-midbrain-cv-invariant

> Midbrain populations have essentially constant CV across all conditions (APN median 0.16 in every condition, MRN 0.20 to 0.21).

In [ ]:
E2M = [("APN each", "APN", paula.CONDITIONS, "each", 0.16, 0.016), ("MRN low", "MRN", paula.CONDITIONS, "min", 0.20, 0.02),
       ("MRN high", "MRN", paula.CONDITIONS, "max", 0.21, 0.021)]
show_expected(check_expected(MED_CV, E2M))

**Result: partially reproduced.** The APN median is 0.146 to 0.172 across conditions, within 0.16 +/- 0.016 in each, and the MRN lowest median (0.185 to 0.186) is within 0.20 +/- 0.02, but the MRN highest (0.233, natural images) is just above 0.21 +/- 0.021. The outcome rests on the unit-selection assumption.

## JS divergence matrices of CV (claims of Fig. 2b)

For each condition, the JS divergence between the CV distributions of every pair of subregions, with 50 and 100 equal histogram bins over the pooled range of each pair, in bits. The sums are within each major area: (all) every pair of its block; (adjacent) the first off-diagonal in Table I order. Percentage differences: (second) |A - B| / B; (mean) |A - B| over the mean of A and B.

In [ ]:
JS_CV = {(rd, nb): {c: paula.js_matrix(w[w.condition == c], "cv", REG, nb) for c in paula.CONDITIONS}
         for rd, w in W.items() for nb in (50, 100)}
def pct_table(JS):
    rows = []
    for (rd, nb), mats in JS.items():
        for sm in ("all", "adjacent"):
            for pm in ("second", "mean"):
                for a in paula.AREAS:
                    s = {c: paula.summed_js(mats[c], a, sm) for c in paula.CONDITIONS}
                    for c1, c2 in paula.COND_PAIRS:
                        rows.append({"reading": rd, "bins": nb, "sum": sm, "pct": pm, "area": a, "pair": f"{c1}-{c2}",
                                     "S1": s[c1], "S2": s[c2], "diff": paula.pct_diff(s[c1], s[c2], pm)})
    return pd.DataFrame(rows)
PCT_CV = pct_table(JS_CV)
print(PCT_CV[(PCT_CV.reading == "a/a") & (PCT_CV.bins == 50) & (PCT_CV.pct == "second")].pivot_table(index=["sum", "area"], columns="pair", values="diff").round(1).to_string())
print("summed JS (reading a/a, 50 bins, all pairs):", {a: {c: round(paula.summed_js(JS_CV[("a/a", 50)][c], a, "all"), 2) for c in paula.CONDITIONS} for a in paula.AREAS})

## fig2b-vis-summed-js-cv

> For CV, the summed within-area JS divergences of visual cortex are nearly equal in active behavior and blank screens (about 1% apart) and differ by about 30% from those of natural images and spontaneous activity.

In [ ]:
FOUR = ["images-active", "active-spont", "images-blanks", "blanks-spont"]
res2b = []
for key, g in PCT_CV[PCT_CV.area == "VIS"].groupby(["reading", "bins", "sum", "pct"]):
    d = g.set_index("pair")["diff"]
    ab = d["active-blanks"]
    four = d[FOUR]
    full = 0.5 <= ab <= 1.5 and bool(((four >= 27) & (four <= 33)).all())
    part = d.idxmin() == "active-blanks" and bool((d.drop("active-blanks") > 15).all())
    res2b.append({"reading": key[0], "bins": key[1], "sum": key[2], "pct": key[3], "active-blanks": round(ab, 2),
                  **{p: round(four[p], 1) for p in FOUR}, "closest pair": d.idxmin(), "reproduced": full, "partial": part})
res2b = pd.DataFrame(res2b)
print(res2b.to_string())
print("combinations reproduced:", int(res2b.reproduced.sum()), "| partial:", int(res2b.partial.sum()), "of", len(res2b))

**Result: partially reproduced.** No combination of readings meets the full rule. Under condition readings a/a and b/a with 50 bins and the sum over all within-area pairs, active and blank screens are the closest pair (0.32% and 0.53% apart) and the other pairs differ by 23% to 40%, which meets the partial rule (4 of 32 combinations); the four pairs against natural images or spontaneous activity are 23% to 40%, not all within 27% to 33%. With 100 bins or the adjacent-pair sum, active and blanks are 3.8% to 9.5% apart and not always the closest pair. The outcome depends on the histogram and sum readings, and rests on the unit-selection assumption.

## fig2b-hp-mb-summed-js-cv

> For CV, the largest hippocampal percentage difference in summed JS divergence is between active behavior and spontaneous activity (about 15%), and midbrain has the smallest differences (at most 7%).

In [ ]:
res2h = []
for key, g in PCT_CV.groupby(["reading", "bins", "sum", "pct"]):
    t = g.pivot_table(index="area", columns="pair", values="diff")
    hp = t.loc["HP"]
    mb_max = t.loc["MB"].max()
    mb_small = int((t.idxmin(axis=0) == "MB").sum())
    ok = tol(hp["active-spont"], 15, 1.5) and hp.idxmax() == "active-spont" and mb_max <= 7.7 and mb_small >= 4
    res2h.append({"reading": key[0], "bins": key[1], "sum": key[2], "pct": key[3], "HP active-spont": round(hp["active-spont"], 1),
                  "HP largest pair": hp.idxmax(), "MB max": round(mb_max, 1), "pairs where MB smallest": mb_small, "reproduced": ok})
res2h = pd.DataFrame(res2h)
print(res2h.to_string())
print("combinations reproduced:", int(res2h.reproduced.sum()), "of", len(res2h))

**Result: not reproduced.** Active against spontaneous is the largest hippocampal difference under every combination, as the paper says, but it is 41% to 71%, not 15% +/- 1.5%, and the largest midbrain difference is 22% to 41%, not at most 7.7%; the midbrain has the smallest difference in at most 4 of the 6 pairs. The outcome rests on the unit-selection assumption.

## fig2b-visually-driven-js-cv-low

> Among the three visually driven conditions, the CV-based JS divergences of cortical regions differ little between conditions (typically less than 4%).

In [ ]:
VDP = ["images-active", "images-blanks", "active-blanks"]
r = PCT_CV[(PCT_CV.area == "VIS") & PCT_CV.pair.isin(VDP)].pivot_table(index=["reading", "bins", "sum", "pct"], columns="pair", values="diff")
r["n at most 4.4%"] = (r[VDP] <= 4.4).sum(axis=1)
r["reproduced"] = r["n at most 4.4%"] >= 2
print(r.round(1).to_string())
print("combinations reproduced (percentage reading):", int(r.reproduced.sum()), "of", len(r))
vis = [x for x in REG if paula.REGION_AREA[x] == "VIS"]
vi = np.triu_indices(len(vis), 1)
for (rd, nb), mats in JS_CV.items():
    fr = {c: float((mats[c].loc[vis, vis].values[vi] <= 0.22).mean()) for c in VD}
    print(f"JS reading, {rd}, {nb} bins: fraction of VIS-VIS pairs at most 0.22: {({k: round(v, 2) for k, v in fr.items()})}; "
          f"VIS-VIS JS range {min(mats[c].loc[vis, vis].values[vi].min() for c in VD):.3f} to {max(mats[c].loc[vis, vis].values[vi].max() for c in VD):.3f}; passes {all(v > 0.5 for v in fr.values())}")

**Result: reproduced.** Under the second reading the criterion names (the legend's JS values), most VIS-VIS divergences are at most 0.22 in each visually driven condition (fractions 0.73 to 1.0) under every condition and histogram reading. Under the percentage reading no combination has two of the three pairs at or below 4.4% (active against blanks is 0.3% to 9.5%, the pairs with natural images 16% to 40%). The outcome rests on the second reading and on the unit-selection assumption.

## fig2b-dg-ca3-high-js-cv

> DG and CA3 have high CV-based JS divergences in all four conditions, largest in spontaneous activity (median 0.84 for both) and blank screens (median 0.63 for both).

In [ ]:
hp_regs = [x for x in REG if paula.REGION_AREA[x] == "HP"]
for variant, others in [("all other subregions", None), ("other hippocampal subregions", hp_regs)]:
    for (rd, nb), mats in JS_CV.items():
        med = {}
        for reg in ["DG", "CA3"]:
            for c in paula.CONDITIONS:
                row = mats[c].loc[reg].drop(reg)
                if others:
                    row = row[[o for o in others if o != reg]]
                med[(reg, c)] = row.median()
        ok = (tol(med[("DG", "spont")], 0.84, 0.084) and tol(med[("CA3", "spont")], 0.84, 0.084) and tol(med[("DG", "blanks")], 0.63, 0.063)
              and tol(med[("CA3", "blanks")], 0.63, 0.063)
              and all(med[(g, "spont")] == max(med[(g, c)] for c in paula.CONDITIONS) for g in ["DG", "CA3"]))
        print(f"{variant}, {rd}, {nb} bins: " + ", ".join(f"{g} {c} {v:.3f}" for (g, c), v in med.items()) + f"; passes {ok}")

**Result: partially reproduced.** The median divergence of DG and of CA3 from the other subregions is largest in spontaneous activity under every reading, and DG's spontaneous median (0.744 to 0.805) is within 0.84 +/- 0.084, but CA3's spontaneous median is 0.450 to 0.544 and the blank-screen medians are 0.461 to 0.489 (DG) and 0.227 to 0.249 (CA3), against 0.84 and 0.63. The hippocampal-only variant gives DG 0.762 to 0.805 and CA3 0.555 to 0.638 in spontaneous activity, with blanks still below 0.63 for CA3. The outcome rests on the unit-selection assumption.

## fig2b-legend-js-cv

> In the CV-based JS divergences, visual and thalamic regions diverge strongly between spontaneous and visually driven states (above 0.40 for most visual areas, above 0.30 for thalamic regions), DG and CA3 moderately (0.15 to 0.35), and midbrain least (below 0.15 for most condition pairs).

Reading (a): per subregion, the JS divergence between its own CV distributions in two conditions. Reading (b): per subregion, the mean absolute difference between its row of the region-pair matrix in spontaneous activity and in a visually driven condition (this notebook's operational form of "comparing a region's row"). A subregion's value for "spontaneous against visually driven" is the mean over the three visually driven conditions.

In [ ]:
def legend_values(rd, nb):
    w = W[rd]
    out = {}
    for reg in REG:
        x = w[w.region == reg]
        out[reg] = {f"{c1}-{c2}": paula.js_divergence(x[x.condition == c1].cv, x[x.condition == c2].cv, nb) for c1, c2 in paula.COND_PAIRS}
    a = pd.DataFrame(out).T
    mats = JS_CV[(rd, nb)]
    b = pd.DataFrame({reg: {f"{c1}-{c2}": float((mats[c1].loc[reg].drop(reg) - mats[c2].loc[reg].drop(reg)).abs().mean()) for c1, c2 in paula.COND_PAIRS} for reg in REG}).T
    return a, b

SVD = ["images-spont", "active-spont", "blanks-spont"]
res_leg = []
for rd in W:
    for nb in (50, 100):
        for lab, t in zip(["a", "b"], legend_values(rd, nb)):
            sv = t[SVD].mean(axis=1)
            v = [x for x in REG if paula.REGION_AREA[x] == "VIS"]
            th = [x for x in REG if paula.REGION_AREA[x] == "TH"]
            mb = [x for x in REG if paula.REGION_AREA[x] == "MB"]
            dgca3 = t.loc[["DG", "CA3"]].values.ravel()
            res_leg.append({"reading": rd, "bins": nb, "measure": lab,
                            "VIS > 0.44": f"{int((sv[v] > 0.44).sum())}/{len(v)}", "TH > 0.33": f"{int((sv[th] > 0.33).sum())}/{len(th)}",
                            "DG, CA3 range": f"{dgca3.min():.3f}-{dgca3.max():.3f}", "MB < 0.165": f"{int((t.loc[mb].values < 0.165).sum())}/{t.loc[mb].size}",
                            "passes": bool((sv[v] > 0.44).mean() > 0.5 and (sv[th] > 0.33).mean() > 0.5 and within(dgca3, 0.135, 0.385).all() and (t.loc[mb].values < 0.165).mean() > 0.5)})
            if rd == "a/a" and nb == 50:
                print(f"measure {lab}, reading a/a, 50 bins:\n", t.round(3).T.to_string())
print(pd.DataFrame(res_leg).to_string())

**Result: not reproduced.** Under reading a, 3 of 6 visual subregions exceed 0.44 (not more than half), at most 1 of 9 thalamic subregions exceeds 0.33, and the DG and CA3 values (0.004 to 0.122) lie below 0.135; under reading b no visual or thalamic subregion passes. Most midbrain values are below 0.165 under reading b, and under reading a with 50 bins. The outcome rests on the unit-selection assumption.

## fig3a-reference-systems

> In the C-H plane with D = 6, white noise lies at H near 1 and C near 0, a sine near H 0 and C near 0.1, exponentially correlated Brownian motion at C 0.3 to 0.5 for H 0.3 to 0.6, and chaotic maps (logistic, tent, Ikeda) near maximum complexity, C 0.4 to 0.5 at H 0.5 to 0.8.

Synthetic series as defined in Supplemental Material Sec. I, 10^5 points after 10^3 dropped (claims file). Correlated Gaussian noise: increments from N(0, 1) noise convolved with the normalized kernel w_k = exp(-k / (K alpha)), K = floor(100 alpha) + 1, then summed (Sec. I.A). The sine is sampled at 1000 points per period (its sampling is not given; a slow sine is the one that reaches H near 0). "Near" a value of 0 is read as within 0.05 (a standard choice; 10% of 0 is no tolerance), and near 1 as at least 0.9. The chaotic maps are judged at their fully chaotic parameters (logistic r = 4, tent mu = 1.9999, Ikeda p = 6) and the sweeps of the open ranges are shown.

In [ ]:
rng3 = np.random.default_rng(zlib.crc32(b"paula-2026/fig3a-reference-systems"))
N3, DROP = 100_000, 1_000
def logistic(r, n=N3 + DROP, x=0.4):
    out = np.empty(n)
    for i in range(n):
        x = r * x * (1 - x); out[i] = x
    return out[DROP:]
def tent(mu, n=N3 + DROP, x=0.3141):
    out = np.empty(n)
    for i in range(n):
        x = mu * x if x < 0.5 else mu * (1 - x); out[i] = x
    return out[DROP:]
def ikeda(p, u=0.9, n=N3 + DROP):
    x = y = 0.1; out = np.empty(n)
    for i in range(n):
        t = p - 6 / (1 + x * x + y * y)
        x, y = 1 + u * (x * np.cos(t) - y * np.sin(t)), u * (x * np.sin(t) + y * np.cos(t)); out[i] = x
    return out[DROP:]
def corr_walk(alpha, n=N3):
    K = int(np.floor(100 * alpha)) + 1
    w = np.exp(-np.arange(K) / (K * alpha)); w /= w.sum()
    e = rng3.standard_normal(n + K)
    return np.cumsum(np.convolve(e, w, mode="valid")[:n])
def fk_noise(k, n=N3):
    f = np.fft.rfftfreq(n); amp = np.zeros_like(f); amp[1:] = f[1:] ** (-k / 2)
    ph = np.exp(2j * np.pi * rng3.random(len(f)))
    return np.fft.irfft(amp * ph, n)
ref = []
ref.append(("white noise", "", *paula.bandt_pompe(rng3.standard_normal(N3))))
ref.append(("sine", "1000 points per period", *paula.bandt_pompe(np.sin(2 * np.pi * np.arange(N3) / 1000))))
ref.append(("Brownian motion", "", *paula.bandt_pompe(np.cumsum(rng3.standard_normal(N3)))))
for a in [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
    ref.append(("exp. correlated walk", f"alpha={a}", *paula.bandt_pompe(corr_walk(a))))
for r in [3.5, 3.6, 3.7, 3.8, 3.9, 4.0]:
    ref.append(("logistic", f"r={r}", *paula.bandt_pompe(logistic(r))))
for mu in [1.2, 1.4, 1.6, 1.8, 1.9999]:
    ref.append(("tent", f"mu={mu}", *paula.bandt_pompe(tent(mu))))
for p in [5, 6, 8, 11, 14, 17]:
    ref.append(("Ikeda", f"p={p}", *paula.bandt_pompe(ikeda(p))))
for k in [0, 0.5, 1, 1.5, 2, 2.5, 3]:
    ref.append(("1/f^k noise", f"k={k}", *paula.bandt_pompe(fk_noise(k))))
REF = pd.DataFrame(ref, columns=["system", "parameter", "H", "C"])
print(REF.round(3).to_string())
g = lambda s, p: REF[(REF.system == s) & (REF.parameter == p)].iloc[0]
wn, sn = g("white noise", ""), g("sine", "1000 points per period")
ok_wn = wn.H >= 0.9 and wn.C <= 0.05
ok_sn = sn.H <= 0.05 and within(sn.C, 0.09, 0.11)
ew = REF[(REF.system == "exp. correlated walk") & within(REF.H, 0.27, 0.66)]
ok_ew = len(ew) > 0 and bool(within(ew.C, 0.27, 0.55).all())
chaos = pd.DataFrame([g("logistic", "r=4.0"), g("tent", "mu=1.9999"), g("Ikeda", "p=6")])
ok_ch = bool((within(chaos.C, 0.36, 0.55) & within(chaos.H, 0.45, 0.88)).all())
print(f"white noise H {wn.H:.3f} C {wn.C:.4f}: {ok_wn}; sine H {sn.H:.3f} C {sn.C:.3f}: {ok_sn}")
print(f"exp. correlated walk: {len(ew)} of 8 alphas with H in 0.27 to 0.66, their C {ew.C.min() if len(ew) else np.nan:.3f} to {ew.C.max() if len(ew) else np.nan:.3f}; all walks H {REF[REF.system=='exp. correlated walk'].H.min():.3f} to {REF[REF.system=='exp. correlated walk'].H.max():.3f}: {ok_ew}")
print("chaotic maps at chaotic parameters:\n", chaos.round(3).to_string(), "\n inside C 0.36 to 0.55 and H 0.45 to 0.88:", ok_ch)
fig, ax = plt.subplots(figsize=(6, 4.5))
for s, gg in REF.groupby("system"):
    ax.plot(gg.H, gg.C, "o-" if len(gg) > 1 else "o", label=s, ms=4)
ax.set_xlabel("H"); ax.set_ylabel("C"); ax.set_xlim(0, 1); ax.set_ylim(0, 0.55); ax.legend(fontsize=7); plt.show()
print("plotted H range", REF.H.min().round(3), REF.H.max().round(3), "C range", REF.C.min().round(3), REF.C.max().round(3))

**Result: partially reproduced.** White noise is at H 0.999, C 0.001, and the logistic map (r = 4) and tent map (mu = 1.9999) are at H 0.63, C 0.48, inside the chaotic range. The sine (1000 points per period) is at H 0.115 and C 0.113, just outside the reading of near 0 and 0.09 to 0.11; the Ikeda map with u = 0.9 lies at H 0.11 to 0.38 for p in [5, 17], not near maximum complexity; and the exponentially correlated walks with H in 0.27 to 0.66 have C 0.233 to 0.334, partly below 0.27. The parameters of Fig. 3a are not in the text, so these positions rest on the parameter readings shown.

## fig3b-complexity-levels-by-region

> Across the hour of active recording, visual cortex and hippocampus keep the highest statistical complexity (C about 0.13 to 0.15), thalamus is intermediate (about 0.11) and midbrain lowest (about 0.10), and these differences stay stable over time.

Reading (a): the 36 active windows; reading (b): the eight natural-image windows per population in image order (alphabetical image names, as no other order is given).

In [ ]:
def fig3b(w, label):
    tc = w.groupby(["area", "k"]).C.agg(["mean", "sem", "size"]).reset_index()
    piv = tc.pivot(index="k", columns="area", values="mean")
    means = piv.mean()
    ok_v = all(within(means[a], 0.117, 0.165) for a in ["VIS", "HP"])
    ok_t, ok_m = tol(means["TH"], 0.11, 0.011), tol(means["MB"], 0.10, 0.01)
    order = (piv.MB < piv.TH) & (piv.TH < piv.VIS) & (piv.TH < piv.HP)
    print(f"{label}: mean over windows {means.round(4).to_dict()}; VIS and HP in 0.117 to 0.165 {ok_v}; TH {ok_t}; MB {ok_m}; "
          f"order MB < TH < VIS and HP at {int(order.sum())} of {len(order)} indices; range over indices {piv.min().round(3).to_dict()} to {piv.max().round(3).to_dict()}; "
          f"largest standard error {tc['sem'].max():.4f}")
    return piv, ok_v and ok_t and ok_m and bool(order.all()), ok_v and ok_t and ok_m and order.mean() > 0.5
piv_a, full_a, part_a = fig3b(ACT, "reading a (active)")
IMG = W["a/a"][W["a/a"].condition == "images"]
piv_b, full_b, part_b = fig3b(IMG, "reading b (images)")
print("variant units=qc:"); _ = fig3b(ACTQ, "reading a, units=qc")
fig, ax = plt.subplots(figsize=(7, 3.5))
for a in paula.AREAS:
    ax.plot(piv_a.index, piv_a[a], label=a)
ax.set_xlabel("active window index"); ax.set_ylabel("mean C"); ax.legend(); plt.show()

**Result: reproduced.** Under reading a (the 36 active windows), the mean C over windows is 0.161 (VIS), 0.163 (HP), 0.109 (TH) and 0.101 (MB), each within its tolerance, and the order MB < TH < VIS and HP holds at all 36 window indices. Under reading b (natural images) hippocampus falls below thalamus (0.108 against 0.117) and the order holds at none of the 8 indices. The 'qc' variant also holds at 36 of 36. The outcome rests on reading a and on the unit-selection assumption.

## C-H plane per area (claims of Figs. 3c to 3f)

Active windows (the reading the claims file assigns from the point counts), with natural-image windows shown. Spans are the 2.5th to 97.5th percentiles (and the 0th to 100th, a variant). The Mann-Whitney test compares C of the windows with C of their surrogates, two-sided, under the five seeds.

In [ ]:
SPAN = {"VIS": ((0.8, 1.0), (0.1, 0.4)), "TH": ((0.75, 0.95), (0.05, 0.25)), "HP": ((0.85, 1.0), (0.05, 0.4)), "MB": ((0.9, 1.0), (0.05, 0.2))}
NPTS = {"VIS": 21084, "TH": 11265, "HP": 13059, "MB": 6296}
def ch_table(w, surr=True):
    rows = []
    for a in paula.AREAS:
        x = w[w.area == a]
        r = {"area": a, "windows": len(x), "paper windows": NPTS[a]}
        for q, (lo, hi) in [("95", (2.5, 97.5)), ("100", (0, 100))]:
            r[f"H span {q}"] = tuple(np.round(np.percentile(x.H, [lo, hi]), 3))
            r[f"C span {q}"] = tuple(np.round(np.percentile(x.C, [lo, hi]), 3))
        if surr:
            r["surrogate median C"] = round(x.s_C.median(), 4)
            r["empirical median C"] = round(x.C.median(), 4)
            for s in SFX:
                r["p" + (s or "")] = stats.mannwhitneyu(x.C, x["s_C" + s]).pvalue
                r["surr med" + s] = x["s_C" + s].median() if s else r["surrogate median C"]
        rows.append(r)
    return pd.DataFrame(rows).set_index("area")
CH = ch_table(ACT)
print(CH.T.to_string())
print("\nnatural-image windows:\n", ch_table(IMG, surr=False).T.to_string())

def judge_ch(a, q="95"):
    (hl, hh), (cl, ch) = SPAN[a]
    H0, H1 = CH.loc[a, f"H span {q}"]; C0, C1 = CH.loc[a, f"C span {q}"]
    hw, cw = widen(hl, hh), widen(cl, ch)
    span_ok = hw[0] <= H0 and H1 <= hw[1] and cw[0] <= C0 and C1 <= cw[1]
    test_ok = CH.loc[a, "p"] < 0.001 and CH.loc[a, "empirical median C"] > CH.loc[a, "surrogate median C"]
    width = {b: CH.loc[b, f"C span {q}"][1] - CH.loc[b, f"C span {q}"][0] for b in paula.AREAS}
    print(f"{a}, span {q}: H {H0}-{H1} within {hw[0]:.3f}-{hw[1]:.3f}, C {C0}-{C1} within {cw[0]:.3f}-{cw[1]:.3f}: {span_ok}; "
          f"Mann-Whitney p {CH.loc[a, 'p']:.1e} (five seeds up to {max(CH.loc[a, 'p' + s] for s in SFX):.1e}), empirical above surrogate {CH.loc[a, 'empirical median C'] > CH.loc[a, 'surrogate median C']}; "
          f"C span widths {({b: round(v, 3) for b, v in width.items()})}")
    return span_ok, test_ok, width

## fig3c-vis-ch-plane

> Visual cortex windows span H 0.8 to 1.0 and C 0.1 to 0.4 in the C-H plane, and differ from ISI-shuffled surrogates, which cluster at lower complexity (C 0.1 to 0.2), Mann-Whitney p < 0.001.

In [ ]:
for q in ["95", "100"]:
    s_ok, t_ok, wd = judge_ch("VIS", q)
sm = [CH.loc["VIS", "surr med" + s] for s in SFX]
print(f"VIS surrogate median C under five seeds {min(sm):.4f} to {max(sm):.4f}; inside 0.09 to 0.22: {all(0.09 <= v <= 0.22 for v in sm)}")

**Result: partially reproduced.** The test holds: C of the visual windows is above that of their surrogates (median 0.152 against 0.0955, Mann-Whitney p = 0 at printed precision under all five seeds), and the surrogate median lies in 0.09 to 0.22. The 2.5th to 97.5th percentile span of H (0.856 to 0.965) is inside 0.72 to 1.10, but that of C starts at 0.089, just below 0.090; the 0th to 100th span misses on H (0.462). There are 31 956 active windows against the paper's 21 084. The outcome rests on the unit-selection assumption.

## fig3d-th-ch-plane

> Thalamic windows occupy a narrower part of the C-H plane than visual cortex (H 0.75 to 0.95, C 0.05 to 0.25), and differ from surrogates (p < 0.001).

In [ ]:
for q in ["95", "100"]:
    s_ok, t_ok, wd = judge_ch("TH", q)
    print(f"  span {q}: TH C span narrower than VIS: {wd['TH'] < wd['VIS']}")

**Result: reproduced.** The thalamic span (H 0.928 to 0.969, C 0.080 to 0.171, 2.5th to 97.5th percentiles) is inside the widened ranges, its C span (0.091) is narrower than that of visual cortex (0.196), and C is above the surrogate's (p = 0 at printed precision, all five seeds). The 0th to 100th span misses on C (up to 0.344). The outcome rests on the percentile reading of a span and on the unit-selection assumption.

## fig3e-hp-ch-plane

> Hippocampal windows have the broadest complexity distribution (C 0.05 to 0.4) at high entropy (H 0.85 to 1.0), and differ from surrogates (p < 0.001).

In [ ]:
for q in ["95", "100"]:
    s_ok, t_ok, wd = judge_ch("HP", q)
    print(f"  span {q}: HP C span widest of the four: {max(wd, key=wd.get) == 'HP'}")

**Result: reproduced.** The hippocampal span (H 0.857 to 0.966, C 0.086 to 0.298) is inside the widened ranges, its C span (0.212) is the widest of the four areas, and C is above the surrogate's (p = 0 at printed precision, all five seeds). The 0th to 100th span misses on H (0.124) and C (0.441, against 0.440). The outcome rests on the percentile reading of a span and on the unit-selection assumption.

## fig3f-mb-ch-plane

> Midbrain windows occupy the most restricted part of the C-H plane, at high entropy (H 0.9 to 1.0) and low complexity (C 0.05 to 0.2), yet differ from surrogates (p < 0.001).

In [ ]:
for q in ["95", "100"]:
    s_ok, t_ok, wd = judge_ch("MB", q)
    print(f"  span {q}: MB C span narrowest of the four: {min(wd, key=wd.get) == 'MB'}")
fig, ax = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for a_, a in zip(ax, paula.AREAS):
    x = ACT[ACT.area == a]
    a_.plot(x.H, x.C, ".", ms=1, alpha=0.3); a_.plot(x.s_H, x.s_C, ".", ms=1, alpha=0.3, color="k")
    a_.set_title(a); a_.set_xlim(0.5, 1); a_.set_ylim(0, 0.5); a_.set_xlabel("H")
ax[0].set_ylabel("C"); plt.show()
print("windows outside the axes (H < 0.5):", int((ACT.H < 0.5).sum()), "| C > 0.5:", int((ACT.C > 0.5).sum()))

**Result: reproduced.** The midbrain span (H 0.935 to 0.970, C 0.076 to 0.157) is inside the widened ranges, its C span (0.081) is the narrowest of the four, and C is above the surrogate's (Mann-Whitney p = 2.3e-194, at most 7.3e-192 over five seeds). Under the 0th to 100th span it misses on H and is not the narrowest. The outcome rests on the percentile reading of a span and on the unit-selection assumption.

## fig3-complexity-range-fractions

> The fractions of windows by complexity range are, for visual cortex, 0.8% in [0.3, 0.4] and 4.9% in [0.0, 0.1]; thalamus 0.6% in [0.2, 0.3] and 24.6% in [0.0, 0.1]; hippocampus 2.7% in [0.3, 0.4] and 9.5% in [0.0, 0.1]; midbrain 45.4% in [0.1, 0.2] and 54.0% in [0.0, 0.1].

In [ ]:
FR = [("VIS", 0.3, 0.4, 0.8), ("VIS", 0.0, 0.1, 4.9), ("TH", 0.2, 0.3, 0.6), ("TH", 0.0, 0.1, 24.6),
      ("HP", 0.3, 0.4, 2.7), ("HP", 0.0, 0.1, 9.5), ("MB", 0.1, 0.2, 45.4), ("MB", 0.0, 0.1, 54.0)]
for lab, w in [("reading a (active)", ACT), ("reading b (images)", IMG), ("active, units=qc", ACTQ)]:
    out, n = [], 0
    for a, lo, hi, v in FR:
        c = w[w.area == a].C
        f = 100 * ((c >= lo) & ((c < hi) | ((hi == 0.4) & (c <= hi)))).mean()
        ok = tol(f, v, 0.1 * v); n += ok
        out.append(f"{a} [{lo},{hi}) {f:.2f}% ({v})" + ("" if ok else " MISS"))
    print(f"{lab}: {n}/8 within tolerance; " + "; ".join(out))

**Result: not reproduced.** None of the eight fractions is within tolerance on the active windows (for example VIS 10.07% in [0, 0.1) against 4.9%, MB 37.30% in [0.1, 0.2) against 45.4%), one on the natural-image windows, and none under the 'qc' variant. The outcome rests on the unit-selection assumption.

## Medians of C by condition (claims of Fig. 4a to 4d)

In [ ]:
MED_C = med_table("C")
print(MED_C.round(3).to_string())

def mw_pairs(w, regs, bonf):
    rows = []
    for r in regs:
        x = w[w.region == r]
        for c1, c2 in paula.COND_PAIRS:
            p = stats.mannwhitneyu(x[x.condition == c1].C, x[x.condition == c2].C).pvalue
            rows.append({"region": r, "pair": f"{c1}-{c2}", "p": min(1.0, p * bonf)})
    return pd.DataFrame(rows)

## fig4a-vis-complexity-by-condition

> All six visual areas have higher median complexity during natural image presentation (0.16 to 0.26) than during spontaneous activity (0.1), with active behavior and blank screens intermediate (0.1 to 0.19), all pairwise comparisons p < 0.001.

In [ ]:
VISR = [r for r in REG if paula.REGION_AREA[r] == "VIS"]
r4a = {}
for rd in W:
    m = MED_C[rd].loc[VISR]
    order = (m.images > m.active) & (m.images > m.blanks) & (m.active > m.spont) & (m.blanks > m.spont)
    ni = within(m.images, *widen(0.16, 0.26)); sp = within(m.spont, 0.09, 0.11); ab = within(m.active, *widen(0.1, 0.19)) & within(m.blanks, *widen(0.1, 0.19))
    for name, bonf in [("uncorrected", 1), ("Bonferroni 36", 36)]:
        mw = mw_pairs(W[rd], VISR, bonf)
        allp = bool((mw.p < 0.001).all())
        r4a[(rd, name)] = bool(order.all() and ni.all() and sp.all() and ab.all() and allp)
        print(f"reading {rd}, {name}: order holds in {int(order.sum())}/6 ({list(order[~order].index)} fail); images medians {m.images.min():.3f}-{m.images.max():.3f} inside {int(ni.sum())}/6; "
              f"spont {m.spont.min():.3f}-{m.spont.max():.3f} inside {int(sp.sum())}/6; active and blanks {min(m.active.min(), m.blanks.min()):.3f}-{max(m.active.max(), m.blanks.max()):.3f} inside {int(ab.sum())}/6; "
              f"tests p < 0.001 {int((mw.p < 0.001).sum())}/36, largest p {mw.p.max():.1e} ({mw.loc[mw.p.idxmax(), 'region']} {mw.loc[mw.p.idxmax(), 'pair']}); passes {r4a[(rd, name)]}")
m = MED_C["a/a"].loc[VISR]
print("medians, reading a/a:\n", m.round(3).to_string())
iq = W["a/a"][W["a/a"].region.isin(VISR)].groupby(["region", "condition"]).C.agg(lambda x: x.quantile(0.75) - x.quantile(0.25)).unstack()
print("IQRs (legend 0.08 to 0.12 in visually driven states):\n", iq.round(3).to_string())
LEG4A = {"VISl": (0.15, 0.26), "VISal": (0.15, 0.25), "VISp": (0.16, 0.19)}
for r, (lo, hi) in LEG4A.items():
    vd = m.loc[r, VD]
    print(f"legend {r} {lo}-{hi}: visually driven medians {vd.min():.3f}-{vd.max():.3f}, inside widened range {bool(within(vd, *widen(lo, hi)).all())}")

**Result: partially reproduced.** All medians fall in the quoted ranges: natural images 0.160 to 0.265, spontaneous 0.094 to 0.098, active and blanks 0.101 to 0.190. The order holds in five of six areas: in VISp the natural-image median (0.178) is below the active one (0.190) and equal to the blank one, and 35 of 36 pairwise tests give p < 0.001 (VISp images against blanks p = 0.46 to 0.53). The legend's narrower ranges for VISl, VISal and VISp are met. The outcome rests on the unit-selection assumption.

## fig4b-thalamus-lgd-lp-modulated

> Of nine thalamic nuclei, only LGd and LP show significant modulation of complexity across conditions (p < 0.001), higher during visually driven states (0.14 to 0.18) than spontaneous activity (0.11 to 0.12), with smaller effects than visual cortex.

The effect is the visually driven minus spontaneous median, taken (i) as the largest visually driven median minus the spontaneous one and (ii) as the mean of the three visually driven medians minus the spontaneous one; the claims file does not fix it.

In [ ]:
THR = [r for r in REG if paula.REGION_AREA[r] == "TH"]
def thal_tests(w, regs):
    rows = []
    for r in regs:
        x = w[w.region == r]
        kw = stats.kruskal(*[x[x.condition == c].C for c in paula.CONDITIONS]).pvalue
        pw = [stats.mannwhitneyu(x[x.condition == c].C, x[x.condition == "spont"].C).pvalue for c in VD]
        rows.append({"region": r, "kruskal p": kw, "largest pairwise p (VD vs spont)": max(pw)})
    return pd.DataFrame(rows).set_index("region")
for rd in W:
    m = MED_C[rd]
    tt = thal_tests(W[rd], THR)
    eff_max = m.loc[:, VD].max(axis=1) - m.spont
    eff_mean = m.loc[:, VD].mean(axis=1) - m.spont
    res = []
    for r in ["LGd", "LP"]:
        vd = m.loc[r, VD]
        text_ok = bool(within(vd, *widen(0.14, 0.18)).all()) and within(m.loc[r, "spont"], *widen(0.11, 0.12))
        leg = {"LP": ((0.12, 0.14), 0.11), "LGd": ((0.12, 0.15), 0.10)}[r]
        leg_ok = bool(within(vd, *widen(*leg[0])).all()) and tol(m.loc[r, "spont"], leg[1], 0.1 * leg[1])
        res.append((r, vd.round(3).to_dict(), round(m.loc[r, "spont"], 3), text_ok, leg_ok, bool((vd > m.loc[r, "spont"]).all()),
                    bool(eff_max[r] < eff_max[VISR].min()), bool(eff_mean[r] < eff_mean[VISR].min())))
    print(f"reading {rd}:\n", tt.to_string(float_format=lambda v: f"{v:.1e}"))
    for x in res:
        print(f"  {x[0]}: VD medians {x[1]}, spont {x[2]}; text values {x[3]}; legend values {x[4]}; VD above spont {x[5]}; effect below every visual area (i) {x[6]}, (ii) {x[7]}")
    print(f"  effects (i) VIS {eff_max[VISR].round(3).to_dict()} TH {eff_max[THR].round(3).to_dict()}")
    print(f"  effects (ii) VIS {eff_mean[VISR].round(3).to_dict()} TH {eff_mean[THR].round(3).to_dict()}")

**Result: reproduced.** LGd and LP are modulated under both test readings (Kruskal-Wallis p = 7e-69 and 3e-63; every visually driven condition against spontaneous p at most 9e-24 and 2e-17 under reading a/a), their visually driven medians (0.113 to 0.152) exceed the spontaneous ones (0.093 to 0.100) and match the legend's values, not the text's (LGd active 0.121 is below 0.126). Their effect is smaller than that of every visual area when the effect is the largest visually driven median minus the spontaneous one (LGd 0.057 to 0.059, LP 0.043 to 0.044 against 0.065 to 0.170), but not when it is the mean of the three (LGd 0.036 against VISrl 0.030); the first is adopted as an assumption. Under Kruskal-Wallis, MGd, SGN, MGm and MGv also reach p < 0.001, so the statement's "only" holds under the pairwise reading alone.

## fig4b-other-thalamus-not-modulated

> The other seven thalamic nuclei (MGd, PoT, SGN, POL, MGm, MGv, PIL) keep uniformly low complexity (0.10 to 0.13) with no significant stimulus dependence.

In [ ]:
SEVEN = ["MGd", "PoT", "SGN", "POL", "MGm", "MGv", "PIL"]
for rd in W:
    m = MED_C[rd].loc[SEVEN]
    tt = thal_tests(W[rd], SEVEN)
    rng_ok = bool(within(m.values, 0.09, 0.143).all())
    print(f"reading {rd}: medians {m.values.min():.3f}-{m.values.max():.3f} inside 0.09 to 0.143 {rng_ok}; "
          f"Kruskal-Wallis p < 0.001 in {list(tt.index[tt['kruskal p'] < 0.001])}; pairwise (all VD vs spont) p < 0.001 in {list(tt.index[tt['largest pairwise p (VD vs spont)'] < 0.001])}")
    print("   largest condition difference in median C:", (m.max(axis=1) - m.min(axis=1)).round(4).to_dict())
    print("   ", tt.to_string(float_format=lambda v: f"{v:.1e}").replace("\n", "\n    "))

**Result: reproduced.** The medians of the seven nuclei are 0.093 to 0.108, inside 0.09 to 0.143, and their largest condition difference is at most 0.006. Under the pairwise Mann-Whitney reading (every visually driven condition against spontaneous), adopted as the paper's test, none reaches p < 0.001 (smallest largest-p 0.004, SGN). Under the Kruskal-Wallis reading four of them (MGd, SGN, MGm, MGv) reach p < 0.001, which would make the claim not reproduced; the outcome rests on the test reading and the unit-selection assumption.

## fig4c-dg-ca3-complexity

> DG and CA3 show elevated complexity across most conditions (DG 0.19 to 0.27, CA3 0.15 to 0.21), peaking during spontaneous activity, opposite to visual cortex, while CA1, POST, ProS and SUB stay lower and stable (0.11 to 0.15).

In [ ]:
for rd in W:
    m = MED_C[rd]
    # (a) text: DG and CA3 inside their range in at least three of four conditions; CA1, POST, ProS, SUB inside 0.11-0.15 in every condition
    a_dg = int(within(m.loc["DG"], *widen(0.19, 0.27)).sum()); a_ca3 = int(within(m.loc["CA3"], *widen(0.15, 0.21)).sum())
    rest = m.loc[["CA1", "POST", "ProS", "SUB"]]
    a_rest = bool(within(rest.values, *widen(0.11, 0.15)).all())
    peak = all(m.loc[g].idxmax() == "spont" for g in ["DG", "CA3"])
    # (b) legend: natural images apart
    b_ok = (tol(m.loc["DG", "images"], 0.12, 0.012) and tol(m.loc["CA3", "images"], 0.11, 0.011) and tol(m.loc["CA1", "images"], 0.10, 0.010)
            and bool(within(m.loc["DG", ["active", "blanks", "spont"]], *widen(0.19, 0.26)).all())
            and bool(within(m.loc["CA3", ["active", "blanks", "spont"]], *widen(0.13, 0.18)).all())
            and bool(within(m.loc["CA1", ["active", "blanks", "spont"]], *widen(0.12, 0.17)).all())
            and bool(within(m.loc[["POST", "ProS", "SUB"]].values, *widen(0.10, 0.13)).all()))
    print(f"reading {rd}: DG {m.loc['DG'].round(3).to_dict()}; CA3 {m.loc['CA3'].round(3).to_dict()}; spontaneous peak {peak}")
    print(f"   text: DG inside in {a_dg}/4, CA3 in {a_ca3}/4, CA1/POST/ProS/SUB all inside {a_rest} (range {rest.values.min():.3f}-{rest.values.max():.3f}); passes {a_dg >= 3 and a_ca3 >= 3 and a_rest and peak}")
    print(f"   legend: passes {b_ok and peak}; CA1 {m.loc['CA1'].round(3).to_dict()}; POST/ProS/SUB {m.loc[['POST','ProS','SUB']].values.min():.3f}-{m.loc[['POST','ProS','SUB']].values.max():.3f}")

**Result: reproduced.** Under the legend's reading with the spontaneous reading b (a/b and b/b), DG natural images 0.115, CA3 0.108 and CA1 0.096 match the legend's low values, DG (0.178 to 0.257), CA3 (0.137 to 0.196) and CA1 (0.120 to 0.163) match their elevated ranges in the other conditions, POST, ProS and SUB lie in 0.097 to 0.136, and DG and CA3 peak in spontaneous activity. Under the spontaneous reading a, CA3's spontaneous median (printed 0.198) lies just above the widened bound 0.198 and the reading fails. The text's reading fails because CA1 and SUB fall below 0.099 in natural images. The outcome rests on the legend reading and the unit-selection assumption.

## fig4d-midbrain-complexity-invariant

> Most midbrain areas (NB, MRN, SCig, APN) keep uniformly low complexity (0.10 to 0.12) across conditions; only NOT shows modest stimulus dependence (0.13 to 0.15 in visually driven states against 0.13 in spontaneous activity).

In [ ]:
LEG4D = {"NB": (0.11, 0.12), "MRN": (0.10, 0.11), "SCig": (0.10, 0.11), "APN": (0.09, 0.09)}
for rd in W:
    m = MED_C[rd]
    four = m.loc[["NB", "MRN", "SCig", "APN"]]
    text_ok = bool(within(four.values, *widen(0.10, 0.12)).all())
    leg_ok = all(within(m.loc[r], *widen(*v)).all() for r, v in LEG4D.items())
    rngs = (m.loc[[r for r in REG if paula.REGION_AREA[r] == "MB"]].max(axis=1) - m.loc[[r for r in REG if paula.REGION_AREA[r] == "MB"]].min(axis=1))
    not_largest = rngs.idxmax() == "NOT"
    print(f"reading {rd}: medians\n{four.round(3).to_string()}\n NOT {m.loc['NOT'].round(3).to_dict()}; ranges across conditions {rngs.round(3).to_dict()}; "
          f"text values {text_ok}; legend values {leg_ok}; NOT range largest {not_largest}; passes text {text_ok and not_largest}, legend {leg_ok and not_largest}")

**Result: reproduced.** Under the legend's values every condition median of NB (0.107 to 0.110), MRN (0.096 to 0.100), SCig (0.097 to 0.102) and APN (0.086 to 0.090) lies in its widened range, and NOT has the largest range across conditions (0.072 to 0.073 against at most 0.004). Under the text's values APN (0.086 to 0.090) falls below 0.09. The outcome rests on the legend reading and the unit-selection assumption.

## JS divergence matrices of C (claims of Figs. 4e to 4h)

As for CV: per condition, JS divergence in bits between the C distributions of every pair of subregions, with 50 and 100 histogram bins.

In [ ]:
JS_C = {(rd, nb): {c: paula.js_matrix(w[w.condition == c], "C", REG, nb) for c in paula.CONDITIONS}
        for rd, w in W.items() for nb in (50, 100)}
AREA_OF = np.array([paula.REGION_AREA[r] for r in REG])
def pair_vals(m, a_set, b_set):
    """Values of pairs with one subregion in a_set and the other in b_set (each unordered pair once)."""
    out = []
    for i, j in zip(*iu):
        ai, aj = AREA_OF[i], AREA_OF[j]
        if (ai in a_set and aj in b_set) or (aj in a_set and ai in b_set):
            out.append(m.values[i, j])
    return np.array(out)
print("JS of C, reading a/a, 50 bins, mean by area pair:")
for c in paula.CONDITIONS:
    m = JS_C[("a/a", 50)][c]
    print(c, {f"{a}-{b}": round(float(pair_vals(m, {a}, {b}).mean()), 3) for a in paula.AREAS for b in paula.AREAS if a <= b})

## fig4e-js-complexity-natural-images

> During natural image presentation the complexity distributions of visual areas are similar to each other (JS below 0.2) but diverge from thalamus and midbrain (JS above 0.4), and DG and CA3 diverge moderately from other regions (JS 0.3 to 0.5).

In [ ]:
for (rd, nb), mats in JS_C.items():
    m = mats["images"]
    vv = pair_vals(m, {"VIS"}, {"VIS"}); vt = pair_vals(m, {"VIS"}, {"TH"}); vm = pair_vals(m, {"VIS"}, {"MB"})
    dg = np.r_[m.loc["DG"].drop("DG").values, m.loc["CA3"].drop(["CA3", "DG"]).values]
    ok = (vv < 0.22).mean() > 0.5 and (vt > 0.36).mean() > 0.5 and (vm > 0.36).mean() > 0.5 and within(dg, 0.27, 0.55).mean() > 0.5
    print(f"{rd}, {nb} bins: VIS-VIS below 0.22 {(vv < 0.22).mean():.2f} (range {vv.min():.3f}-{vv.max():.3f}); VIS-TH above 0.36 {(vt > 0.36).mean():.2f} (median {np.median(vt):.3f}); "
          f"VIS-MB above 0.36 {(vm > 0.36).mean():.2f} (median {np.median(vm):.3f}); DG/CA3 pairs in 0.27-0.55 {within(dg, 0.27, 0.55).mean():.2f} (median {np.median(dg):.3f}); passes {ok}")

**Result: partially reproduced.** In natural images most VIS-VIS divergences are below 0.22 (0.73 to 0.87), and most VIS-TH (0.74) and VIS-MB (0.63 to 0.67) divergences exceed 0.36, but only 0.20 of the pairs of DG or CA3 with other subregions lie in 0.27 to 0.55 (median 0.161 to 0.176). The outcome rests on the unit-selection assumption.

## fig4fg-js-complexity-active-blanks

> During active behavior and blank screens visual cortex keeps complexity distributions distinct from subcortical structures (JS above 0.4), with within-hippocampus divergences of 0.2 to 0.4 during active behavior.

In [ ]:
for (rd, nb), mats in JS_C.items():
    f = {c: pair_vals(mats[c], {"VIS"}, {"TH", "HP", "MB"}) for c in ["active", "blanks"]}
    hh = pair_vals(mats["active"], {"HP"}, {"HP"})
    ok = all((v > 0.36).mean() > 0.5 for v in f.values()) and within(hh, 0.18, 0.44).mean() > 0.5
    print(f"{rd}, {nb} bins: VIS-subcortical above 0.36, active {(f['active'] > 0.36).mean():.2f} (median {np.median(f['active']):.3f}), blanks {(f['blanks'] > 0.36).mean():.2f} (median {np.median(f['blanks']):.3f}); "
          f"HP-HP active in 0.18-0.44 {within(hh, 0.18, 0.44).mean():.2f} (range {hh.min():.3f}-{hh.max():.3f}); passes {ok}")

**Result: not reproduced.** Fewer than half of the pairs of a visual subregion with a subcortical one exceed 0.36 (0.38 in active behavior, 0.29 to 0.31 in blanks; medians 0.193 to 0.259), and 0.27 to 0.33 of the hippocampal pairs in active behavior lie in 0.18 to 0.44. The outcome rests on the unit-selection assumption.

## fig4h-js-complexity-spontaneous-converges

> In spontaneous activity the JS divergences between regions are lower overall, visual cortex differing less from other regions (JS 0.2 to 0.4) than in visually driven states, and midbrain divergences stay low in all conditions (below 0.2 within the midbrain group).

In [ ]:
for (rd, nb), mats in JS_C.items():
    allm = {c: mats[c].values[iu].mean() for c in paula.CONDITIONS}
    vn = {c: pair_vals(mats[c], {"VIS"}, {"TH", "HP", "MB"}) for c in paula.CONDITIONS}
    mm = {c: pair_vals(mats[c], {"MB"}, {"MB"}) for c in paula.CONDITIONS}
    c1 = all(allm["spont"] < allm[c] for c in VD)
    c2 = all(vn["spont"].mean() < vn[c].mean() for c in VD) and within(vn["spont"], 0.18, 0.44).mean() > 0.5
    c3 = all((mm[c] < 0.22).mean() > 0.5 for c in paula.CONDITIONS)
    print(f"{rd}, {nb} bins: mean of all pairs {({c: round(v, 3) for c, v in allm.items()})} (spont lowest {c1}); "
          f"mean VIS-other {({c: round(float(v.mean()), 3) for c, v in vn.items()})}, spont pairs in 0.18-0.44 {within(vn['spont'], 0.18, 0.44).mean():.2f} ({c2}); "
          f"MB-MB below 0.22 {({c: round(float((v < 0.22).mean()), 2) for c, v in mm.items()})} ({c3}); passes {c1 and c2 and c3}")

**Result: not reproduced.** The mean visual-to-other divergence is lowest in spontaneous activity (0.208 to 0.262 against 0.267 to 0.511), as the statement says, but the mean of all pairs is not (0.264 to 0.332 in spontaneous against 0.218 to 0.310 in the others), only 0.27 to 0.35 of the spontaneous visual-to-other pairs lie in 0.18 to 0.44, and midbrain pairs are below 0.22 in at most half of them in spontaneous activity. The outcome rests on the unit-selection assumption.

## supp-robustness-parameters

> The hierarchical, stimulus-dependent pattern of complexity across regions holds for other parameter choices (bin sizes 20 and 50 ms, D from 3 to 6, tau from 1 to 4).

The visually driven minus spontaneous difference of an area is the median C of all its visually driven windows minus the median of its spontaneous windows (this notebook's operational form).

In [ ]:
w = W["a/a"]
print("windows per condition (the 20 ms series use the same windows):", w.condition.value_counts().to_dict(), "| supplement's 20 ms counts: images 11 321, active 51 452, blanks 11 265, spontaneous 2856")
nok = 0
for bw, d, tau in paula.ROBUST + [(0.05, 6, 1)]:
    col = "C" if (bw, d, tau) == (0.05, 6, 1) else f"C_{int(bw*1000)}_{d}_{tau}"
    m = w.groupby(["region", "condition"])[col].median().unstack()
    vis_ok = bool((m.loc[VISR, "images"] > m.loc[VISR, "spont"]).all())
    diff = {a: w[(w.area == a) & w.condition.isin(VD)][col].median() - w[(w.area == a) & (w.condition == "spont")][col].median() for a in paula.AREAS}
    ok = vis_ok and diff["VIS"] > diff["MB"]
    if col != "C":
        nok += ok
    print(f"bin {int(bw*1000)} ms, D={d}, tau={tau}{' (main)' if col == 'C' else ''}: images above spontaneous in {int((m.loc[VISR, 'images'] > m.loc[VISR, 'spont']).sum())}/6 visual areas; "
          f"VD minus spont by area {({a: round(v, 4) for a, v in diff.items()})}; holds {ok}")
print("combinations of the supplement where the pattern holds:", nok, "of 5")

**Result: partially reproduced.** The pattern (natural images above spontaneous in every visual area, and a larger visually driven minus spontaneous difference in visual cortex than in midbrain) holds for 4 of the 5 combinations; with bin 20 ms, D = 3, tau = 4 natural images exceed spontaneous in 2 of 6 visual areas and the visual difference is -0.0002. The outcome rests on this notebook's form of the area difference and the unit-selection assumption.

## Complexity against CV (claims of Fig. 5)

Per area and condition, windows binned by CV, the mean C and its standard error per bin. Readings: bin width 0.1 or 0.05, with no minimum count or a minimum of 10 windows per bin. Spans are the occupied bins (lower edge of the first to upper edge of the last) and the range of bin means. Reading a/a of the conditions; the condition readings change the spontaneous and blank windows only slightly (medians above).

In [ ]:
TRAJ = {}
for width in (0.1, 0.05):
    for mn in (0, 10):
        for a in paula.AREAS:
            for c in paula.CONDITIONS:
                x = W["a/a"][(W["a/a"].area == a) & (W["a/a"].condition == c)]
                t = paula.trajectory(x.cv, x.C, width, mn)
                mono = bool((np.diff(t["mean"].values) >= -(t.se.values[1:] + t.se.values[:-1])).all())
                rho = stats.spearmanr(x.cv, x.C).statistic
                slope = stats.linregress(x.cv, x.C).slope
                TRAJ[(width, mn, a, c)] = {"cv lo": t.lo.min(), "cv hi": t.hi.max(), "C lo": t["mean"].min(), "C hi": t["mean"].max(),
                                           "bins": len(t), "monotonic": mono, "spearman": rho, "slope": slope, "table": t}
TR = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk != "table"} for k, v in TRAJ.items()}).T
TR.index.names = ["width", "min", "area", "condition"]
print(TR.astype({"cv lo": float, "cv hi": float, "C lo": float, "C hi": float, "spearman": float, "slope": float}).round(3).to_string())
fig, ax = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for a_, c in zip(ax, paula.CONDITIONS):
    for a in paula.AREAS:
        t = TRAJ[(0.1, 10, a, c)]["table"]
        a_.errorbar((t.lo + t.hi) / 2, t["mean"], yerr=t.se, label=a, marker="o", ms=3)
    a_.set_title(c); a_.set_xlabel("CV"); a_.set_xlim(0, 2)
ax[0].set_ylabel("mean C"); ax[0].legend(); plt.show()
print("bins with CV above the axis (2):", int(sum((TRAJ[(0.1, 10, a, c)]["table"].lo >= 2).sum() for a in paula.AREAS for c in paula.CONDITIONS)))

def span_ok(x, lo, hi):
    return tol(x[0], lo, 0.1 * abs(lo)) and tol(x[1], hi, 0.1 * abs(hi))

## fig5-vis-complexity-cv-coupling

> In visual cortex complexity rises with CV during visually driven conditions (natural images CV 0.1 to 0.9 and C 0.08 to 0.34; active behavior CV 0.2 to 1.6 and C 0.10 to 0.38; blanks CV 0.1 to 1.2 and C 0.10 to 0.39), while in spontaneous activity the trajectory compresses, with C below about 0.15.

In [ ]:
Q5 = {"images": ((0.1, 0.9), (0.08, 0.34)), "active": ((0.2, 1.6), (0.10, 0.38)), "blanks": ((0.1, 1.2), (0.10, 0.39))}
for width in (0.1, 0.05):
    for mn in (0, 10):
        oks = []
        for c, (cvr, cr) in Q5.items():
            r = TRAJ[(width, mn, "VIS", c)]
            s1, s2 = span_ok((r["cv lo"], r["cv hi"]), *cvr), span_ok((r["C lo"], r["C hi"]), *cr)
            rise = r["monotonic"] and r["spearman"] > 0
            oks.append(s1 and s2 and rise)
            print(f"width {width}, min {mn}, {c}: CV {r['cv lo']:.2f}-{r['cv hi']:.2f} (quoted {cvr}) {s1}; C {r['C lo']:.3f}-{r['C hi']:.3f} (quoted {cr}) {s2}; "
                  f"monotonic {r['monotonic']}, Spearman {r['spearman']:.3f}")
        sp = TRAJ[(width, mn, "VIS", "spont")]
        print(f"width {width}, min {mn}: spontaneous maximum of bin means {sp['C hi']:.3f} (below 0.165: {sp['C hi'] < 0.165}); passes {all(oks) and sp['C hi'] < 0.165}")

**Result: partially reproduced.** Complexity rises with CV in visual cortex in the three visually driven conditions (Spearman 0.62 to 0.83, and monotonic under bins of 0.1 with at least 10 windows), but no reading matches the spans: for example the natural-image bins reach CV 1.05 to 1.30 against 0.9, and the spontaneous maximum of the bin means is 0.180 to 0.357, not below 0.165. The outcome rests on the unit-selection assumption.

## fig5-th-weaker-coupling

> Thalamus shows a weaker but consistent positive complexity-CV relationship in the visually driven conditions, with shallower slopes than visual cortex and a restricted range (CV 0.2 to 0.8, C 0.10 to 0.20).

In [ ]:
for width in (0.1, 0.05):
    for mn in (0, 10):
        oks = []
        for c in VD:
            r, v = TRAJ[(width, mn, "TH", c)], TRAJ[(width, mn, "VIS", c)]
            s = 0.18 <= r["cv lo"] and r["cv hi"] <= 0.88 and 0.09 <= r["C lo"] and r["C hi"] <= 0.22
            oks.append(r["slope"] > 0 and r["slope"] < v["slope"] and s)
            print(f"width {width}, min {mn}, {c}: TH slope {r['slope']:.4f} vs VIS {v['slope']:.4f}; TH CV {r['cv lo']:.2f}-{r['cv hi']:.2f}, C {r['C lo']:.3f}-{r['C hi']:.3f}; spans inside {s}")
        print(f"  passes {all(oks)}")

**Result: partially reproduced.** In each visually driven condition the thalamic slope is positive and below the visual one (0.084 to 0.111 against 0.216 to 0.350), but the thalamic spans are wider than CV 0.18 to 0.88 and C 0.09 to 0.22 under every reading (for example CV 0.10 to 1.40 in active behavior with at least 10 windows per bin). The outcome rests on the unit-selection assumption.

## fig5-hp-spontaneous-elevated

> Hippocampus shows modest complexity (C 0.12 to 0.19) in the visually driven conditions but markedly elevated complexity during spontaneous activity (C 0.12 to 0.25) at intermediate CV (0.4 to 0.7), comparable to visual cortex during image viewing.

In [ ]:
for width in (0.1, 0.05):
    for mn in (0, 10):
        t = TRAJ[(width, mn, "HP", "spont")]["table"]
        mid = t[(t.lo >= 0.4 - 1e-9) & (t.hi <= 0.7 + 1e-9)]
        top = mid["mean"].max() if len(mid) else np.nan
        vdmax = {c: TRAJ[(width, mn, "HP", c)]["C hi"] for c in VD}
        ok = tol(top, 0.25, 0.025) and all(top > v for v in vdmax.values()) and all(tol(v, 0.19, 0.019) for v in vdmax.values())
        print(f"width {width}, min {mn}: HP spontaneous bin means at CV 0.4-0.7 up to {top:.3f}; HP visually driven maxima {({c: round(v, 3) for c, v in vdmax.items()})}; passes {ok}")

**Result: not reproduced.** The hippocampal spontaneous bin means at CV 0.4 to 0.7 reach 0.238 to 0.244, within 0.25 +/- 0.025, but they do not exceed the hippocampal maxima of the visually driven conditions (active 0.324 to 0.410), which are not within 0.19 +/- 0.019. The outcome rests on the unit-selection assumption.

## fig5-mb-flat

> Midbrain complexity stays low (C 0.10 to 0.13) across the full range of CV (0.1 to 0.9) in all conditions, giving nearly flat trajectories.

In [ ]:
for width in (0.1, 0.05):
    for mn in (0, 10):
        inside = {c: bool(TRAJ[(width, mn, "MB", c)]["C lo"] >= 0.09 and TRAJ[(width, mn, "MB", c)]["C hi"] <= (0.165 if c == "images" else 0.143)) for c in paula.CONDITIONS}
        slopes = {c: {a: abs(TRAJ[(width, mn, a, c)]["slope"]) for a in paula.AREAS} for c in paula.CONDITIONS}
        flat = {c: min(s, key=s.get) == "MB" for c, s in slopes.items()}
        print(f"width {width}, min {mn}: MB bin means {({c: (round(TRAJ[(width, mn, 'MB', c)]['C lo'], 3), round(TRAJ[(width, mn, 'MB', c)]['C hi'], 3)) for c in paula.CONDITIONS})}; "
              f"inside {inside}; MB slope smallest {flat}; |slopes| {({c: {a: round(v, 3) for a, v in s.items()} for c, s in slopes.items()})}; passes {all(inside.values()) and all(flat.values())}")

**Result: partially reproduced.** The midbrain slope of C on CV is the smallest of the four areas in all four conditions (0.042 to 0.072), but its bin means leave 0.09 to 0.143 in three conditions under every reading (for example 0.085 to 0.170 in active behavior with at least 10 windows per bin). The outcome rests on the unit-selection assumption.

## supp-fig6-mean-correlation

> In visual cortex the mean pairwise spike count correlation is elevated during active behavior and blank screens (median 0.08 to 0.13) and near zero during natural images and spontaneous activity (0.00 to 0.01); midbrain values are low and invariant.

In [ ]:
for rd in W:
    for col in ["corr50", "corr20"]:
        m = W[rd].groupby(["region", "condition"])[col].median().unstack()
        v = m.loc[VISR]
        ab = v[["active", "blanks"]].values; ns = v[["images", "spont"]].values
        mb = m.loc[[r for r in REG if paula.REGION_AREA[r] == "MB"]]
        mbr = (mb.max(axis=1) - mb.min(axis=1))
        ok = bool(within(ab, *widen(0.08, 0.13)).all() and (ns < 0.011).all() and (mbr < 0.02).all())
        print(f"{rd}, {col}: VIS active/blanks {ab.min():.3f}-{ab.max():.3f} inside 0.072-0.143 {int(within(ab, *widen(0.08, 0.13)).sum())}/12; "
              f"images/spont {ns.min():.4f}-{ns.max():.4f} below 0.011 {int((ns < 0.011).sum())}/12; MB range across conditions {mbr.round(4).to_dict()}; passes {ok}")
        if rd == "a/a" and col == "corr50":
            print(m.loc[VISR + list(mb.index)].round(4).to_string())

**Result: not reproduced.** The visual medians in active behavior and blanks are 0.014 to 0.052 (50 ms) and 0.007 to 0.029 (20 ms), below 0.072, and the natural-image and spontaneous medians are 0.005 to 0.037, half or more of them above 0.011. Midbrain medians vary by less than 0.02 across conditions except NOT (0.024 to 0.025 at 50 ms). The outcome rests on the unit-selection assumption.

## complexity-separates-images-from-spontaneous

> Neither CV nor mean pairwise correlation separates natural image presentation from spontaneous activity in visual cortex, while statistical complexity does.

In [ ]:
for rd in W:
    w = W[rd]
    for nb in (50, 100):
        rows = []
        for r in VISR:
            x = w[w.region == r]
            a, b = x[x.condition == "images"], x[x.condition == "spont"]
            row = {"region": r}
            for col in ["C", "cv", "corr50"]:
                row[f"JS {col}"] = paula.js_divergence(a[col], b[col], nb)
                qa, qb = a[col].quantile([0.25, 0.75]).values, b[col].quantile([0.25, 0.75]).values
                row[f"IQR overlap {col}"] = bool(qa[0] <= qb[1] and qb[0] <= qa[1])
            rows.append(row)
        t = pd.DataFrame(rows).set_index("region")
        js_ok = ((t["JS C"] > t["JS cv"]) & (t["JS C"] > t["JS corr50"])).sum()
        iqr_ok = ((~t["IQR overlap C"]) & t["IQR overlap cv"] & t["IQR overlap corr50"]).sum()
        print(f"{rd}, {nb} bins: complexity separates while CV and correlation do not, JS reading {js_ok}/6, IQR reading {iqr_ok}/6")
        if rd == "a/a" and nb == 50:
            print(t.round(3).to_string())

**Result: reproduced.** Under the JS reading, in all six visual subregions the divergence between natural images and spontaneous activity is larger for complexity (0.443 to 0.849, reading a/a, 50 bins) than for CV (0.198 to 0.544) and mean correlation (0.142 to 0.434), under every condition and histogram reading. Under the IQR reading it holds in 1 or 2 of 6, since the CV and correlation IQRs often do not overlap either. The outcome rests on the JS reading and the unit-selection assumption.

## Summary

| Claim | Outcome |
|---|---|
| table1-sample | not reproduced |
| fig1c-cv-region-medians | not reproduced (assumed) |
| fig1c-cv-above-surrogate | partially reproduced (assumed) |
| fig1d-surrogate-abolishes-cv-correlations | not reproduced (assumed) |
| fig1e-positive-cv-modules | partially reproduced (assumed) |
| fig1e-dg-ca3-anticorrelation-hubs | partially reproduced (assumed) |
| fig1e-midbrain-weakest-cv-correlations | partially reproduced (assumed) |
| fig2a-vis-cv-by-condition | reproduced (assumed) |
| fig2a-thalamus-cv-by-condition | reproduced (assumed) |
| fig2a-hippocampus-cv-stable | partially reproduced (assumed) |
| fig2a-midbrain-cv-invariant | partially reproduced (assumed) |
| fig2b-vis-summed-js-cv | partially reproduced (assumed) |
| fig2b-hp-mb-summed-js-cv | not reproduced (assumed) |
| fig2b-visually-driven-js-cv-low | reproduced (assumed) |
| fig2b-dg-ca3-high-js-cv | partially reproduced (assumed) |
| fig2b-legend-js-cv | not reproduced (assumed) |
| fig3a-reference-systems | partially reproduced (synthetic series) |
| fig3b-complexity-levels-by-region | reproduced (assumed) |
| fig3c-vis-ch-plane | partially reproduced (assumed) |
| fig3d-th-ch-plane | reproduced (assumed) |
| fig3e-hp-ch-plane | reproduced (assumed) |
| fig3f-mb-ch-plane | reproduced (assumed) |
| fig3-complexity-range-fractions | not reproduced (assumed) |
| fig4a-vis-complexity-by-condition | partially reproduced (assumed) |
| fig4b-thalamus-lgd-lp-modulated | reproduced (assumed) |
| fig4b-other-thalamus-not-modulated | reproduced (assumed) |
| fig4c-dg-ca3-complexity | reproduced (assumed) |
| fig4d-midbrain-complexity-invariant | reproduced (assumed) |
| fig4e-js-complexity-natural-images | partially reproduced (assumed) |
| fig4fg-js-complexity-active-blanks | not reproduced (assumed) |
| fig4h-js-complexity-spontaneous-converges | not reproduced (assumed) |
| supp-robustness-parameters | partially reproduced (assumed) |
| fig5-vis-complexity-cv-coupling | partially reproduced (assumed) |
| fig5-th-weaker-coupling | partially reproduced (assumed) |
| fig5-hp-spontaneous-elevated | not reproduced (assumed) |
| fig5-mb-flat | partially reproduced (assumed) |
| supp-fig6-mean-correlation | not reproduced (assumed) |
| complexity-separates-images-from-spontaneous | reproduced (assumed) |

"Assumed" marks an outcome that rests on a choice the paper does not make; the first of them, for every claim on the recordings, is the unit selection: all units of the units table, the reading that keeps Table I's 26 subregions, since neither reading of the claims file reproduces the paper's counts.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper's 102 sessions are not identified, and every one of the 153 ecephys sessions of the dandiset meets both session readings, so all 153 are used; the populations and windows are about 1.5 times the paper's. The unit filter is not stated; neither reading gives Table I's neuron counts. The surrogate of each window is drawn once under the seed of `fig1c-cv-above-surrogate` (and four more) and serves every claim on surrogates. The JS histograms, spans, the blank and spontaneous windows, the test of "modulation" and several effect sizes are readings named in the claims file or in the claim sections above, each shown.

What was found about the dataset. The 153 sessions (81 mice) each hold an active block of 3599 to 3611 s, a 289-s gray screen after it and eight images; no recording appears twice (no shared spike-time fingerprint, spike count, or subject and start time). The visual cortex responds to every session's flash onsets and not to onsets shifted into the gray screen or to omitted flashes, so the spike times and the stimulus table share a clock in all 153 sessions. The blank series are about 287 s long (median), not the "approximately 240 s" of Methods II.A. The location labels include generic ones (`TH`, `MB`, `HPF`) for units the atlas did not assign to a subregion.